<a href="https://colab.research.google.com/github/ubermanproductionsva-jpg/helix-rna-folding-engine/blob/main/arc_v10_beam_object_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 ARC V10 — Beam + Object Reasoning (Colab Ready)

This notebook is **mobile-safe** and **ready to run**:
- Creates folders
- Creates/loads ARC task JSON
- Implements object extraction
- Adds DSL + beam search
- Runs solver and prints result

👉 Run cells top-to-bottom.

In [1]:
# 📁 Setup folders
import os, json
os.makedirs('data/raw', exist_ok=True)
print('Folders ready')

Folders ready


In [2]:
# 📦 Create demo ARC task (skip if you already uploaded your own)
demo = {
  'demo_task': {
    'train': [
      {'input': [[1,0],[0,1]], 'output': [[2,0],[0,2]]}
    ],
    'test': [
      {'input': [[1,0],[0,1]]}
    ]
  }
}

with open('data/raw/arc_tasks.json', 'w') as f:
    json.dump(demo, f)

print('Demo task saved to data/raw/arc_tasks.json')

Demo task saved to data/raw/arc_tasks.json


In [3]:
# ============================================
# ARC CONTROLLER-INTEGRATED SOLVER LOOP (V1.0)
# Colab-ready single cell
# ============================================

from __future__ import annotations

import copy
import json
import math
import statistics
import traceback
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Tuple, Callable

# -------------------------------------------------------------------
# ASSUMPTION:
# The following components already exist in the notebook/runtime:
#   - train_task_map
#   - compositional grammar engine
#   - primitive + object + tiling rules
#   - pipeline generator
#   - scoring system
#   - grammar controller (ANTI-IDENTITY ENABLED)
#   - memory DB
#   - trace builder
#
# This cell wraps and integrates them into:
#   solve_task(task, task_id=None)
# and:
#   solve_dataset(tasks)
#
# No global side effects are required for memory writes unless you pass
# write_memory=True. When write_memory=False, memory DB is not mutated.
# -------------------------------------------------------------------


# ============================================================
# CONFIG
# ============================================================

DEFAULT_SOLVER_CONFIG = {
    "memory_threshold": 0.85,
    "memory_write_threshold": 0.98,
    "candidate_limit": 256,
    "identity_penalty": 0.08,
    "prefer_exact_bonus": 0.25,
    "prefer_compositional_bonus": 0.05,
    "debug": True,
}


# ============================================================
# SAFE UTILITIES
# ============================================================

def _deepcopy_or_same(x):
    try:
        return copy.deepcopy(x)
    except Exception:
        return x


def _safe_mean(values: List[float], default: float = 0.0) -> float:
    vals = [v for v in values if isinstance(v, (int, float))]
    return float(sum(vals) / len(vals)) if vals else float(default)


def _clip01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))


def _to_tuple_grid(grid: List[List[int]]) -> Tuple[Tuple[int, ...], ...]:
    return tuple(tuple(int(v) for v in row) for row in grid)


def _is_grid(x: Any) -> bool:
    return (
        isinstance(x, list)
        and len(x) >= 0
        and all(isinstance(r, list) for r in x)
        and all(all(isinstance(v, int) for v in r) for r in x)
    )


def _grid_shape(grid: List[List[int]]) -> Tuple[int, int]:
    if not _is_grid(grid):
        return (0, 0)
    h = len(grid)
    w = len(grid[0]) if h > 0 else 0
    return (h, w)


def _same_shape(a: List[List[int]], b: List[List[int]]) -> bool:
    return _grid_shape(a) == _grid_shape(b)


def _exact_grid_match(a: List[List[int]], b: List[List[int]]) -> bool:
    return _to_tuple_grid(a) == _to_tuple_grid(b)


def _grid_colors(grid: List[List[int]]) -> List[int]:
    if not _is_grid(grid):
        return []
    return sorted({v for row in grid for v in row})


def _count_nonzero(grid: List[List[int]]) -> int:
    if not _is_grid(grid):
        return 0
    return sum(1 for row in grid for v in row if v != 0)


def _find_connected_components(grid: List[List[int]]) -> List[Dict[str, Any]]:
    """
    Simple 4-connected non-zero same-color components.
    Only used for signature extraction, not as a solver primitive.
    """
    if not _is_grid(grid) or not grid:
        return []

    h, w = len(grid), len(grid[0])
    seen = [[False] * w for _ in range(h)]
    comps = []

    for r in range(h):
        for c in range(w):
            if seen[r][c] or grid[r][c] == 0:
                continue

            color = grid[r][c]
            stack = [(r, c)]
            seen[r][c] = True
            cells = []

            while stack:
                rr, cc = stack.pop()
                cells.append((rr, cc))
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = rr + dr, cc + dc
                    if 0 <= nr < h and 0 <= nc < w and not seen[nr][nc] and grid[nr][nc] == color:
                        seen[nr][nc] = True
                        stack.append((nr, nc))

            rs = [p[0] for p in cells]
            cs = [p[1] for p in cells]
            comps.append({
                "color": color,
                "size": len(cells),
                "bbox": (min(rs), min(cs), max(rs), max(cs)),
                "cells": cells,
            })

    return comps


def _infer_scale_factor(inp: List[List[int]], out: List[List[int]]) -> Tuple[float, float]:
    ih, iw = _grid_shape(inp)
    oh, ow = _grid_shape(out)
    return (
        float(oh / ih) if ih else 0.0,
        float(ow / iw) if iw else 0.0,
    )


def _looks_like_identity_rule(rule_template: Any) -> bool:
    if rule_template is None:
        return False
    if isinstance(rule_template, str):
        s = rule_template.lower()
        return s in {"identity", "copy", "noop", "pass_through", "pass-through"}
    if isinstance(rule_template, (list, tuple)):
        joined = " ".join(str(x).lower() for x in rule_template)
        return "identity" in joined or joined.strip() in {"copy", "noop"}
    return False


def _has_composition(rule_template: Any) -> bool:
    if isinstance(rule_template, (list, tuple)):
        return len(rule_template) >= 2
    if isinstance(rule_template, str):
        s = rule_template.lower()
        return ("+" in s) or ("->" in s) or ("compose" in s) or ("then" in s)
    return False


def _safe_jsonable(x: Any) -> Any:
    try:
        json.dumps(x)
        return x
    except Exception:
        if isinstance(x, dict):
            return {str(k): _safe_jsonable(v) for k, v in x.items()}
        if isinstance(x, (list, tuple)):
            return [_safe_jsonable(v) for v in x]
        return repr(x)


def _call_if_exists(name: str, *args, **kwargs):
    fn = globals().get(name)
    if fn is None or not callable(fn):
        return None
    return fn(*args, **kwargs)


# ============================================================
# DATA STRUCTURES
# ============================================================

@dataclass
class Candidate:
    rule_template: Any
    parameters: Dict[str, Any] = field(default_factory=dict)
    train_outputs: List[Optional[List[List[int]]]] = field(default_factory=list)
    pair_scores: List[float] = field(default_factory=list)
    avg_score: float = 0.0
    exact_all: bool = False
    exact_count: int = 0
    controller_score: float = 0.0
    source: str = "reasoning"
    explanation: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


# ============================================================
# TASK NORMALIZATION
# ============================================================

def normalize_task(task: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(task, dict):
        raise TypeError("task must be a dict")

    train = task.get("train", [])
    test = task.get("test", [])

    if not isinstance(train, list) or not isinstance(test, list):
        raise ValueError("task must contain 'train' and 'test' as lists")

    for i, pair in enumerate(train):
        if not isinstance(pair, dict) or "input" not in pair or "output" not in pair:
            raise ValueError(f"train[{i}] must contain 'input' and 'output'")
        if not _is_grid(pair["input"]) or not _is_grid(pair["output"]):
            raise ValueError(f"train[{i}] input/output must be integer grids")

    for i, pair in enumerate(test):
        if not isinstance(pair, dict) or "input" not in pair:
            raise ValueError(f"test[{i}] must contain 'input'")
        if not _is_grid(pair["input"]):
            raise ValueError(f"test[{i}] input must be an integer grid")

    return task


# ============================================================
# STEP 1: TASK SIGNATURE
# ============================================================

def extract_task_signature(task: Dict[str, Any]) -> Dict[str, Any]:
    task = normalize_task(task)

    train = task["train"]
    test = task["test"]

    input_shapes = [_grid_shape(p["input"]) for p in train]
    output_shapes = [_grid_shape(p["output"]) for p in train]
    input_color_sets = [_grid_colors(p["input"]) for p in train]
    output_color_sets = [_grid_colors(p["output"]) for p in train]
    input_nonzero = [_count_nonzero(p["input"]) for p in train]
    output_nonzero = [_count_nonzero(p["output"]) for p in train]
    input_obj_counts = [len(_find_connected_components(p["input"])) for p in train]
    output_obj_counts = [len(_find_connected_components(p["output"])) for p in train]
    scale_factors = [_infer_scale_factor(p["input"], p["output"]) for p in train]

    signature = {
        "n_train": len(train),
        "n_test": len(test),
        "train_input_shapes": input_shapes,
        "train_output_shapes": output_shapes,
        "train_input_color_sets": input_color_sets,
        "train_output_color_sets": output_color_sets,
        "train_input_nonzero": input_nonzero,
        "train_output_nonzero": output_nonzero,
        "train_input_object_counts": input_obj_counts,
        "train_output_object_counts": output_obj_counts,
        "train_scale_factors": scale_factors,
        "test_input_shapes": [_grid_shape(p["input"]) for p in test],
        "shape_delta_patterns": [
            (
                output_shapes[i][0] - input_shapes[i][0],
                output_shapes[i][1] - input_shapes[i][1]
            )
            for i in range(len(train))
        ],
        "shape_ratio_patterns": scale_factors,
        "input_color_union": sorted({c for colors in input_color_sets for c in colors}),
        "output_color_union": sorted({c for colors in output_color_sets for c in colors}),
    }

    return signature


# ============================================================
# MEMORY SUPPORT
# ============================================================

def _default_memory_db() -> Dict[str, Any]:
    return {
        "entries": []
    }


def _get_memory_entries(memory_db: Optional[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if memory_db is None:
        return []
    if isinstance(memory_db, dict) and isinstance(memory_db.get("entries"), list):
        return memory_db["entries"]
    if isinstance(memory_db, list):
        return memory_db
    return []


def _jaccard_similarity(a: List[Any], b: List[Any]) -> float:
    sa, sb = set(map(str, a)), set(map(str, b))
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def _shape_list_similarity(a: List[Tuple[int, int]], b: List[Tuple[int, int]]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    n = min(len(a), len(b))
    exact = sum(1 for i in range(n) if tuple(a[i]) == tuple(b[i]))
    denom = max(len(a), len(b))
    return exact / denom if denom else 0.0


def _numeric_list_similarity(a: List[float], b: List[float]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    n = min(len(a), len(b))
    sims = []
    for i in range(n):
        av = float(a[i])
        bv = float(b[i])
        denom = max(abs(av), abs(bv), 1.0)
        sims.append(1.0 - min(abs(av - bv) / denom, 1.0))
    return _safe_mean(sims, default=0.0)


def _flatten_pairs(xs: List[Tuple[float, float]]) -> List[float]:
    out = []
    for a, b in xs:
        out.append(float(a))
        out.append(float(b))
    return out


def signature_similarity(sig_a: Dict[str, Any], sig_b: Dict[str, Any]) -> float:
    """
    Heuristic similarity over task signatures.
    """
    pieces = []

    pieces.append(_shape_list_similarity(
        sig_a.get("train_input_shapes", []),
        sig_b.get("train_input_shapes", []),
    ))
    pieces.append(_shape_list_similarity(
        sig_a.get("train_output_shapes", []),
        sig_b.get("train_output_shapes", []),
    ))
    pieces.append(_numeric_list_similarity(
        sig_a.get("train_input_object_counts", []),
        sig_b.get("train_input_object_counts", []),
    ))
    pieces.append(_numeric_list_similarity(
        sig_a.get("train_output_object_counts", []),
        sig_b.get("train_output_object_counts", []),
    ))
    pieces.append(_numeric_list_similarity(
        sig_a.get("train_input_nonzero", []),
        sig_b.get("train_input_nonzero", []),
    ))
    pieces.append(_numeric_list_similarity(
        sig_a.get("train_output_nonzero", []),
        sig_b.get("train_output_nonzero", []),
    ))
    pieces.append(_numeric_list_similarity(
        _flatten_pairs(sig_a.get("train_scale_factors", [])),
        _flatten_pairs(sig_b.get("train_scale_factors", [])),
    ))
    pieces.append(_jaccard_similarity(
        sig_a.get("input_color_union", []),
        sig_b.get("input_color_union", []),
    ))
    pieces.append(_jaccard_similarity(
        sig_a.get("output_color_union", []),
        sig_b.get("output_color_union", []),
    ))

    return _clip01(_safe_mean(pieces, default=0.0))


def retrieve_from_memory(
    task: Dict[str, Any],
    memory_db: Optional[Dict[str, Any]] = None,
    memory_threshold: float = 0.85,
    debug: bool = True,
) -> Dict[str, Any]:
    signature = extract_task_signature(task)
    entries = _get_memory_entries(memory_db)

    best_entry = None
    best_score = -1.0

    for entry in entries:
        try:
            entry_sig = entry.get("signature", {})
            sim = signature_similarity(signature, entry_sig)
            if sim > best_score:
                best_score = sim
                best_entry = entry
        except Exception:
            continue

    hit = best_entry is not None and best_score >= memory_threshold

    if debug:
        if hit:
            print(f"[MEMORY] HIT | similarity={best_score:.4f} | threshold={memory_threshold:.4f}")
        else:
            print(f"[MEMORY] MISS | best_similarity={max(best_score, 0.0):.4f} | threshold={memory_threshold:.4f}")

    return {
        "signature": signature,
        "hit": hit,
        "similarity": max(best_score, 0.0),
        "entry": _deepcopy_or_same(best_entry) if best_entry is not None else None,
    }


def write_to_memory_db(
    memory_db: Optional[Dict[str, Any]],
    signature: Dict[str, Any],
    best_candidate: Candidate,
    task_id: Optional[str] = None,
    debug: bool = True,
) -> Dict[str, Any]:
    """
    Explicit write only. Returns the mutated/new DB.
    """
    db = _deepcopy_or_same(memory_db) if memory_db is not None else _default_memory_db()
    if isinstance(db, list):
        db = {"entries": db}
    if "entries" not in db or not isinstance(db["entries"], list):
        db["entries"] = []

    entry = {
        "task_id": task_id,
        "signature": _safe_jsonable(signature),
        "rule_template": _safe_jsonable(best_candidate.rule_template),
        "parameters": _safe_jsonable(best_candidate.parameters),
        "score": best_candidate.controller_score if best_candidate.controller_score else best_candidate.avg_score,
        "avg_score": best_candidate.avg_score,
        "exact_all": best_candidate.exact_all,
        "explanation": best_candidate.explanation,
        "source": best_candidate.source,
    }
    db["entries"].append(entry)

    if debug:
        print(f"[MEMORY WRITE] stored task_id={task_id} rule={best_candidate.rule_template}")

    return db


# ============================================================
# EXECUTION WRAPPERS
# ============================================================

def execute_pipeline_on_grid(rule_template: Any, parameters: Dict[str, Any], grid: List[List[int]]) -> List[List[int]]:
    """
    Try existing runtime executors first.
    Expected existing components may expose one of:
      - execute_compositional_grammar(best_candidate_dict, grid)
      - execute_compositional_grammar(rule_template, grid, parameters=...)
      - apply_pipeline(grid, rule_template, parameters)
      - execute_pipeline(rule_template, parameters, grid)
    """
    # 1) execute_compositional_grammar(candidate_dict, grid)
    fn = globals().get("execute_compositional_grammar")
    if callable(fn):
        try:
            return fn({
                "rule_template": rule_template,
                "parameters": parameters,
            }, grid)
        except TypeError:
            pass
        except Exception:
            pass

        try:
            return fn(rule_template, grid, parameters=parameters)
        except TypeError:
            pass
        except Exception:
            pass

        try:
            return fn(rule_template, parameters, grid)
        except TypeError:
            pass
        except Exception:
            pass

    # 2) apply_pipeline(grid, rule_template, parameters)
    fn = globals().get("apply_pipeline")
    if callable(fn):
        try:
            return fn(grid, rule_template, parameters)
        except Exception:
            pass

    # 3) execute_pipeline(rule_template, parameters, grid)
    fn = globals().get("execute_pipeline")
    if callable(fn):
        try:
            return fn(rule_template, parameters, grid)
        except Exception:
            pass

    # 4) direct identity fallback only if explicitly identity-like
    if _looks_like_identity_rule(rule_template):
        return _deepcopy_or_same(grid)

    raise RuntimeError(
        "Could not execute pipeline. Expected one of "
        "execute_compositional_grammar / apply_pipeline / execute_pipeline "
        "to exist with a compatible signature."
    )


def score_prediction(pred: List[List[int]], target: List[List[int]]) -> float:
    """
    Exact-match oriented scoring. Tries existing scoring fn first, else uses:
    - 1.0 for exact match
    - partial overlap score if same shape
    - 0.0 otherwise
    """
    fn = globals().get("score_grid_pair") or globals().get("score_prediction")
    if callable(fn) and fn is not score_prediction:
        try:
            s = fn(pred, target)
            if isinstance(s, (int, float)):
                return _clip01(float(s))
        except Exception:
            pass

    if not _is_grid(pred) or not _is_grid(target):
        return 0.0

    if _exact_grid_match(pred, target):
        return 1.0

    if not _same_shape(pred, target):
        return 0.0

    total = len(pred) * len(pred[0]) if pred and pred[0] else 0
    if total == 0:
        return 0.0

    match = 0
    for r in range(len(pred)):
        for c in range(len(pred[0])):
            if pred[r][c] == target[r][c]:
                match += 1
    return match / total


# ============================================================
# STEP 3: CANDIDATE GENERATION
# ============================================================

def _normalize_candidate(raw: Any) -> Candidate:
    if isinstance(raw, Candidate):
        return raw

    if isinstance(raw, dict):
        return Candidate(
            rule_template=raw.get("rule_template", raw.get("pipeline", raw.get("rule"))),
            parameters=_deepcopy_or_same(raw.get("parameters", {})) or {},
            source=raw.get("source", "reasoning"),
            explanation=raw.get("explanation", ""),
            metadata=_deepcopy_or_same(raw.get("metadata", {})) or {},
        )

    if isinstance(raw, (list, tuple, str)):
        return Candidate(
            rule_template=_deepcopy_or_same(raw),
            parameters={},
            source="reasoning",
            explanation="Candidate normalized from raw template.",
            metadata={},
        )

    raise TypeError(f"Unsupported candidate type: {type(raw)}")


def generate_candidate_pipelines(
    task: Dict[str, Any],
    candidate_limit: int = 256,
    debug: bool = True,
) -> List[Candidate]:
    """
    Try the existing generator first.
    Falls back to a tiny safe seed set if unavailable.
    """
    task = normalize_task(task)
    raw_candidates = None

    for name in ["generate_candidate_pipelines", "generate_pipelines", "pipeline_generator"]:
        fn = globals().get(name)
        if callable(fn) and fn is not generate_candidate_pipelines:
            try:
                raw_candidates = fn(task)
                break
            except Exception:
                continue

    candidates: List[Candidate] = []

    if raw_candidates is not None:
        try:
            for raw in raw_candidates:
                candidates.append(_normalize_candidate(raw))
        except Exception as e:
            raise RuntimeError(f"Candidate generator returned unreadable results: {e}")
    else:
        # Safe seed fallback.
        # This is not meant to replace your actual generator.
        seed_templates = [
            "identity",
            ["extract_largest_object"],
            ["extract_smallest_object"],
            ["color_map"],
            ["tile_n"],
            ["crop_nonzero_bbox"],
            ["extract_largest_object", "color_map"],
            ["crop_nonzero_bbox", "tile_n"],
        ]
        candidates = [Candidate(rule_template=t, parameters={}, source="reasoning") for t in seed_templates]

    # Deduplicate
    seen = set()
    deduped = []
    for cand in candidates:
        key = json.dumps({
            "rule_template": _safe_jsonable(cand.rule_template),
            "parameters": _safe_jsonable(cand.parameters),
        }, sort_keys=True)
        if key not in seen:
            seen.add(key)
            deduped.append(cand)

    deduped = deduped[:candidate_limit]

    if debug:
        print(f"[CANDIDATES] generated={len(deduped)}")

    return deduped


# ============================================================
# STEP 4: SCORING
# ============================================================

def evaluate_candidate_on_train(task: Dict[str, Any], candidate: Candidate) -> Candidate:
    task = normalize_task(task)
    out = _deepcopy_or_same(candidate)
    out.train_outputs = []
    out.pair_scores = []

    for pair in task["train"]:
        try:
            pred = execute_pipeline_on_grid(out.rule_template, out.parameters, pair["input"])
            out.train_outputs.append(pred)
            out.pair_scores.append(score_prediction(pred, pair["output"]))
        except Exception:
            out.train_outputs.append(None)
            out.pair_scores.append(0.0)

    out.avg_score = _safe_mean(out.pair_scores, default=0.0)
    out.exact_count = sum(1 for s in out.pair_scores if s == 1.0)
    out.exact_all = (out.exact_count == len(task["train"])) if task["train"] else False

    if not out.explanation:
        out.explanation = (
            f"Scored across {len(task['train'])} train pair(s): "
            f"avg_score={out.avg_score:.4f}, exact_count={out.exact_count}/{len(task['train'])}."
        )

    return out


def score_candidates(task: Dict[str, Any], candidates: List[Candidate], debug: bool = True) -> List[Candidate]:
    scored = [evaluate_candidate_on_train(task, cand) for cand in candidates]

    scored.sort(
        key=lambda c: (
            c.exact_all,
            c.exact_count,
            c.avg_score,
            not _looks_like_identity_rule(c.rule_template),
            _has_composition(c.rule_template),
        ),
        reverse=True,
    )

    if debug:
        print("[SCORING] top candidates:")
        for i, cand in enumerate(scored[:10]):
            print(
                f"  {i+1:02d}. rule={cand.rule_template} | "
                f"avg={cand.avg_score:.4f} | exact_all={cand.exact_all} | exact_count={cand.exact_count}"
            )

    return scored


# ============================================================
# STEP 5: CONTROLLER DECISION
# ============================================================

def _controller_adjusted_score(candidate: Candidate, config: Dict[str, Any]) -> float:
    score = float(candidate.avg_score)

    if candidate.exact_all:
        score += float(config.get("prefer_exact_bonus", 0.25))

    if _has_composition(candidate.rule_template):
        score += float(config.get("prefer_compositional_bonus", 0.05))

    if _looks_like_identity_rule(candidate.rule_template) and not candidate.exact_all:
        score -= float(config.get("identity_penalty", 0.08))

    return score


def grammar_controller_select(
    task: Dict[str, Any],
    candidates: List[Candidate],
    config: Optional[Dict[str, Any]] = None,
    debug: bool = True,
) -> Candidate:
    """
    Uses existing grammar_controller if available.
    Otherwise applies anti-identity + exact-match-first policy.
    """
    cfg = dict(DEFAULT_SOLVER_CONFIG)
    if config:
        cfg.update(config)

    external = globals().get("grammar_controller")
    if callable(external):
        try:
            raw = external(task, [c.to_dict() for c in candidates])
            best = _normalize_candidate(raw)

            # Preserve scored attributes if external controller didn't return them.
            lookup = {}
            for c in candidates:
                key = json.dumps({
                    "rule_template": _safe_jsonable(c.rule_template),
                    "parameters": _safe_jsonable(c.parameters),
                }, sort_keys=True)
                lookup[key] = c
            best_key = json.dumps({
                "rule_template": _safe_jsonable(best.rule_template),
                "parameters": _safe_jsonable(best.parameters),
            }, sort_keys=True)
            if best_key in lookup:
                base = lookup[best_key]
                if best.avg_score == 0.0:
                    best.avg_score = base.avg_score
                if not best.pair_scores:
                    best.pair_scores = base.pair_scores
                if best.exact_count == 0:
                    best.exact_count = base.exact_count
                if not best.exact_all:
                    best.exact_all = base.exact_all
                if not best.explanation:
                    best.explanation = base.explanation
                if not best.metadata:
                    best.metadata = base.metadata
                if not best.source:
                    best.source = base.source

            best.controller_score = _controller_adjusted_score(best, cfg)

            if debug:
                print(
                    f"[CONTROLLER] external decision | rule={best.rule_template} | "
                    f"avg={best.avg_score:.4f} | controller_score={best.controller_score:.4f}"
                )
            return best
        except Exception as e:
            if debug:
                print(f"[CONTROLLER] external controller failed, using internal fallback: {e}")

    # Internal fallback policy
    ranked = []
    for c in candidates:
        cc = _deepcopy_or_same(c)
        cc.controller_score = _controller_adjusted_score(cc, cfg)
        ranked.append(cc)

    ranked.sort(
        key=lambda c: (
            c.exact_all,
            c.controller_score,
            c.exact_count,
            c.avg_score,
        ),
        reverse=True,
    )

    best = ranked[0]
    if debug:
        print(
            f"[CONTROLLER] fallback decision | rule={best.rule_template} | "
            f"avg={best.avg_score:.4f} | controller_score={best.controller_score:.4f}"
        )
    return best


# ============================================================
# STEP 6: TRACE
# ============================================================

def build_solver_trace(
    task_id: Optional[str],
    task: Dict[str, Any],
    signature: Dict[str, Any],
    mode: str,
    memory_result: Dict[str, Any],
    candidates: List[Candidate],
    best: Candidate,
) -> Dict[str, Any]:
    """
    Try existing trace builder first. Falls back to structured trace dict.
    """
    external = globals().get("build_compositional_trace")
    if callable(external):
        try:
            trace = external(
                task_id,
                task,
                {
                    "rule_template": best.rule_template,
                    "parameters": best.parameters,
                    "score": best.controller_score if best.controller_score else best.avg_score,
                    "avg_score": best.avg_score,
                    "exact_all": best.exact_all,
                    "explanation": best.explanation,
                }
            )
            if isinstance(trace, dict):
                trace.setdefault("solver_mode", mode)
                trace.setdefault("memory_similarity", memory_result.get("similarity", 0.0))
                return trace
        except Exception:
            pass

    return {
        "task_id": task_id,
        "solver_mode": mode,
        "signature": _safe_jsonable(signature),
        "memory": {
            "hit": memory_result.get("hit", False),
            "similarity": memory_result.get("similarity", 0.0),
            "matched_task_id": (
                memory_result.get("entry", {}).get("task_id")
                if memory_result.get("entry") else None
            ),
        },
        "candidate_summary": [
            {
                "rule_template": _safe_jsonable(c.rule_template),
                "parameters": _safe_jsonable(c.parameters),
                "avg_score": c.avg_score,
                "exact_all": c.exact_all,
                "exact_count": c.exact_count,
                "controller_score": c.controller_score,
                "source": c.source,
            }
            for c in candidates[:20]
        ],
        "selected": {
            "rule_template": _safe_jsonable(best.rule_template),
            "parameters": _safe_jsonable(best.parameters),
            "avg_score": best.avg_score,
            "exact_all": best.exact_all,
            "controller_score": best.controller_score,
            "explanation": best.explanation,
        },
    }


# ============================================================
# STEP 7: TEST EXECUTION
# ============================================================

def execute_selected_pipeline_on_tests(task: Dict[str, Any], best: Candidate) -> List[List[List[int]]]:
    outputs = []
    for pair in task["test"]:
        pred = execute_pipeline_on_grid(best.rule_template, best.parameters, pair["input"])
        outputs.append(pred)
    return outputs


# ============================================================
# MAIN SOLVER
# ============================================================

def solve_task(
    task: Dict[str, Any],
    task_id: Optional[str] = None,
    memory_db: Optional[Dict[str, Any]] = None,
    config: Optional[Dict[str, Any]] = None,
    write_memory: bool = False,
    return_updated_memory: bool = False,
) -> Dict[str, Any]:
    """
    End-to-end ARC solver loop.

    Returns a structured dict with:
      - task_id
      - mode
      - pipeline
      - parameters
      - score
      - explanation
      - trace
      - test_output
      - updated_memory_db (optional)
    """
    cfg = dict(DEFAULT_SOLVER_CONFIG)
    if config:
        cfg.update(config)

    debug = bool(cfg.get("debug", True))

    try:
        task = normalize_task(task)

        # STEP 1 + 2: signature + memory
        memory_result = retrieve_from_memory(
            task=task,
            memory_db=memory_db,
            memory_threshold=float(cfg["memory_threshold"]),
            debug=debug,
        )
        signature = memory_result["signature"]

        mode = "reasoning"
        candidates: List[Candidate] = []

        # Strong memory path
        if memory_result["hit"]:
            entry = memory_result["entry"]
            best = Candidate(
                rule_template=entry.get("rule_template"),
                parameters=_deepcopy_or_same(entry.get("parameters", {})) or {},
                source="memory",
                explanation=entry.get(
                    "explanation",
                    f"Retrieved from memory with similarity={memory_result['similarity']:.4f}."
                ),
                metadata={"memory_similarity": memory_result["similarity"]},
            )
            best = evaluate_candidate_on_train(task, best)
            best.controller_score = _controller_adjusted_score(best, cfg)
            mode = "memory"

            if debug:
                print(f"[SOLVER] mode=memory | rule={best.rule_template}")

            candidates = [best]

        else:
            # STEP 3: candidate generation
            candidates = generate_candidate_pipelines(
                task=task,
                candidate_limit=int(cfg["candidate_limit"]),
                debug=debug,
            )

            # STEP 4: scoring
            candidates = score_candidates(task, candidates, debug=debug)

            # STEP 5: controller decision
            best = grammar_controller_select(
                task=task,
                candidates=candidates,
                config=cfg,
                debug=debug,
            )

            if debug:
                print(f"[SOLVER] mode=reasoning | selected={best.rule_template}")

        # STEP 6: trace
        trace = build_solver_trace(
            task_id=task_id,
            task=task,
            signature=signature,
            mode=mode,
            memory_result=memory_result,
            candidates=candidates,
            best=best,
        )

        # STEP 6b: optional explicit memory write
        updated_memory_db = memory_db
        should_write = (
            write_memory
            and best.avg_score >= float(cfg["memory_write_threshold"])
            and not _looks_like_identity_rule(best.rule_template)
        )

        if should_write:
            updated_memory_db = write_to_memory_db(
                memory_db=memory_db,
                signature=signature,
                best_candidate=best,
                task_id=task_id,
                debug=debug,
            )
        elif debug:
            print(
                "[MEMORY WRITE] skipped | "
                f"write_memory={write_memory} | avg_score={best.avg_score:.4f} | "
                f"identity={_looks_like_identity_rule(best.rule_template)}"
            )

        # STEP 7: test execution
        test_output = execute_selected_pipeline_on_tests(task, best)

        # STEP 8: structured return
        result = {
            "task_id": task_id,
            "mode": mode,
            "pipeline": _safe_jsonable(best.rule_template),
            "parameters": _safe_jsonable(best.parameters),
            "score": float(best.controller_score if best.controller_score else best.avg_score),
            "avg_score": float(best.avg_score),
            "exact_all": bool(best.exact_all),
            "explanation": best.explanation,
            "trace": _safe_jsonable(trace),
            "test_output": _safe_jsonable(test_output),
        }

        if return_updated_memory:
            result["updated_memory_db"] = _safe_jsonable(updated_memory_db)

        return result

    except Exception as e:
        err_trace = traceback.format_exc()
        print("[SOLVER ERROR]")
        print(err_trace)

        return {
            "task_id": task_id,
            "mode": "error",
            "pipeline": None,
            "parameters": {},
            "score": 0.0,
            "avg_score": 0.0,
            "exact_all": False,
            "explanation": f"Solver failed: {type(e).__name__}: {e}",
            "trace": {
                "task_id": task_id,
                "solver_mode": "error",
                "error_type": type(e).__name__,
                "error_message": str(e),
                "traceback": err_trace,
            },
            "test_output": [],
            **({"updated_memory_db": _safe_jsonable(memory_db)} if return_updated_memory else {})
        }


# ============================================================
# BONUS: DATASET SOLVER
# ============================================================

def solve_dataset(
    tasks: Dict[str, Dict[str, Any]],
    memory_db: Optional[Dict[str, Any]] = None,
    config: Optional[Dict[str, Any]] = None,
    write_memory: bool = False,
) -> Dict[str, Any]:
    """
    Runs solve_task across a dict of tasks:
      {task_id: task_dict, ...}

    Returns:
      - results_by_task
      - summary
      - updated_memory_db
    """
    if not isinstance(tasks, dict):
        raise TypeError("tasks must be a dict mapping task_id -> task")

    cfg = dict(DEFAULT_SOLVER_CONFIG)
    if config:
        cfg.update(config)

    debug = bool(cfg.get("debug", True))
    current_memory = _deepcopy_or_same(memory_db) if memory_db is not None else _default_memory_db()

    results_by_task = {}
    n_total = 0
    n_memory = 0
    n_reasoning = 0
    n_error = 0
    train_exact_all = 0

    for task_id, task in tasks.items():
        result = solve_task(
            task=task,
            task_id=task_id,
            memory_db=current_memory,
            config=cfg,
            write_memory=write_memory,
            return_updated_memory=True,
        )
        results_by_task[task_id] = result
        current_memory = result.get("updated_memory_db", current_memory)

        n_total += 1
        mode = result.get("mode", "error")
        if mode == "memory":
            n_memory += 1
        elif mode == "reasoning":
            n_reasoning += 1
        else:
            n_error += 1

        if result.get("exact_all", False):
            train_exact_all += 1

    summary = {
        "n_tasks": n_total,
        "train_exact_all_count": train_exact_all,
        "train_exact_all_rate": (train_exact_all / n_total) if n_total else 0.0,
        "memory_usage_count": n_memory,
        "memory_usage_rate": (n_memory / n_total) if n_total else 0.0,
        "reasoning_count": n_reasoning,
        "reasoning_rate": (n_reasoning / n_total) if n_total else 0.0,
        "error_count": n_error,
        "error_rate": (n_error / n_total) if n_total else 0.0,
    }

    if debug:
        print("[DATASET SUMMARY]")
        for k, v in summary.items():
            print(f"  {k}: {v}")

    return {
        "results_by_task": results_by_task,
        "summary": summary,
        "updated_memory_db": current_memory,
    }


# ============================================================
# OPTIONAL SUBMISSION HELPER FOR ARC-AGI-2 FORMAT
# ============================================================
# ARC-AGI-2 requires exactly 2 attempts per test output in submission.json :contentReference[oaicite:0]{index=0}

def make_arc_agi_2_submission(results_by_task: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    """
    Convert solve_dataset output into ARC-AGI-2 submission structure.

    This emits identical attempt_1 / attempt_2 by default.
    You can later extend this to emit a second diverse candidate.
    """
    submission = {}
    for task_id, result in results_by_task.items():
        test_outputs = result.get("test_output", [])
        submission[task_id] = []
        for grid in test_outputs:
            submission[task_id].append({
                "attempt_1": grid,
                "attempt_2": grid,
            })
    return submission


# ============================================================
# QUICK USAGE
# ============================================================
#
# single_result = solve_task(
#     task=my_task,
#     task_id="abc123",
#     memory_db=memory_db,
#     config={"debug": True, "memory_threshold": 0.85},
#     write_memory=True,
#     return_updated_memory=True,
# )
#
# dataset_result = solve_dataset(
#     tasks=train_task_map,
#     memory_db=memory_db,
#     config={"debug": True},
#     write_memory=True,
# )
#
# submission = make_arc_agi_2_submission(dataset_result["results_by_task"])
# with open("submission.json", "w") as f:
#     json.dump(submission, f)
#
# ============================================================

In [4]:
# 🧠 ARC Solver (Beam + Object Ops)
from collections import deque

Grid = list[list[int]]

def get_objects(grid: Grid):
    h, w = len(grid), len(grid[0])
    visited = set()
    objects = []

    for r in range(h):
        for c in range(w):
            if (r, c) in visited or grid[r][c] == 0:
                continue

            color = grid[r][c]
            queue = deque([(r, c)])
            visited.add((r, c))
            pixels = []

            while queue:
                x, y = queue.popleft()
                pixels.append((x, y))

                for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < h and 0 <= ny < w:
                        if (nx, ny) not in visited and grid[nx][ny] == color:
                            visited.add((nx, ny))
                            queue.append((nx, ny))

            objects.append({'color': color, 'pixels': pixels, 'area': len(pixels)})

    return objects

def color_map(grid: Grid, mapping: dict[int, int]):
    return [[mapping.get(v, v) for v in row] for row in grid]

def rotate(grid: Grid):
    return [list(row) for row in zip(*grid[::-1])]

def reflect(grid: Grid):
    return [row[::-1] for row in grid]

def tile_n(grid: Grid, n: int):
    return [row * n for row in grid for _ in range(n)]

def recolor_largest(grid: Grid, color: int):
    objs = get_objects(grid)
    if not objs:
        return grid
    obj = max(objs, key=lambda o: o['area'])
    out = [row[:] for row in grid]
    for r, c in obj['pixels']:
        out[r][c] = color
    return out

def run_program(grid: Grid, program):
    g = [row[:] for row in grid]
    for op, args in program:
        if op == 'color_map':
            g = color_map(g, args)
        elif op == 'rotate':
            g = rotate(g)
        elif op == 'reflect':
            g = reflect(g)
        elif op == 'tile':
            g = tile_n(g, args)
        elif op == 'recolor_largest':
            g = recolor_largest(g, args)
    return g

def score(pred: Grid, target: Grid):
    if len(pred) != len(target):
        return 0.0
    total = len(target) * len(target[0])
    correct = sum(1 for r in range(len(target)) for c in range(len(target[0])) if pred[r][c] == target[r][c])
    return correct / total if total else 0.0

def infer_color_map(inp: Grid, out: Grid):
    mapping = {}
    for r in range(len(inp)):
        for c in range(len(inp[0])):
            mapping[inp[r][c]] = out[r][c]
    return mapping

OPS = [
    ('rotate', None),
    ('reflect', None),
    ('tile', 2),
    ('tile', 3),
    ('recolor_largest', 1),
    ('recolor_largest', 2),
    ('recolor_largest', 3),
]

def beam_search(task, width=5, depth=3):
    train = task['train'][0]
    inp, out = train['input'], train['output']

    programs = []
    mapping = infer_color_map(inp, out)
    programs.append([('color_map', mapping)])

    beam = programs

    for _ in range(depth):
        candidates = []
        for prog in beam:
            for op, arg in OPS:
                new_prog = prog + [(op, arg)]
                pred = run_program(inp, new_prog)
                sc = score(pred, out)
                candidates.append((sc, new_prog))

        candidates.sort(key=lambda x: x[0], reverse=True)
        beam = [p for _, p in candidates[:width]]

        if candidates and candidates[0][0] == 1.0:
            return candidates[0][1]

    return beam[0] if beam else []

# ---- Run solver ----
with open('data/raw/arc_tasks.json') as f:
    data = json.load(f)

task = data[list(data.keys())[0]]
program = beam_search(task)

print('🏆 Best Program:', program)

test_input = task['test'][0]['input']
output = run_program(test_input, program)

print('\nInput:')
print(test_input)
print('\nOutput:')
print(output)

🏆 Best Program: [('color_map', {1: 2, 0: 0}), ('recolor_largest', 2)]

Input:
[[1, 0], [0, 1]]

Output:
[[2, 0], [0, 2]]


In [5]:
# ============================================
# UNZIP ALL ZIP FILES IN CURRENT DIRECTORY
# ============================================

import os
import zipfile

# Get current working directory (this is your file pane location)
base_path = os.getcwd()

print(f"[INFO] Scanning directory: {base_path}")

# Find all zip files
zip_files = [f for f in os.listdir(base_path) if f.endswith(".zip")]

if not zip_files:
    print("[INFO] No zip files found.")
else:
    print(f"[INFO] Found {len(zip_files)} zip file(s):")
    for z in zip_files:
        print(f"  - {z}")

# Extract each zip
for zip_name in zip_files:
    zip_path = os.path.join(base_path, zip_name)

    # Create folder with same name (without .zip)
    extract_folder = os.path.join(base_path, zip_name.replace(".zip", ""))

    print(f"\n[EXTRACTING] {zip_name} → {extract_folder}")

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)
        print(f"[SUCCESS] Extracted: {zip_name}")
    except Exception as e:
        print(f"[ERROR] Failed to extract {zip_name}: {e}")

print("\n[DONE] All zip files processed.")

[INFO] Scanning directory: /content
[INFO] Found 2 zip file(s):
  - drive-download-20260330T005331Z-3-001.zip
  - arc_tasks-20260330T005352Z-3-001.zip

[EXTRACTING] drive-download-20260330T005331Z-3-001.zip → /content/drive-download-20260330T005331Z-3-001
[SUCCESS] Extracted: drive-download-20260330T005331Z-3-001.zip

[EXTRACTING] arc_tasks-20260330T005352Z-3-001.zip → /content/arc_tasks-20260330T005352Z-3-001
[SUCCESS] Extracted: arc_tasks-20260330T005352Z-3-001.zip

[DONE] All zip files processed.


In [12]:
# ============================================================
# ARC CLEAN SEMANTIC ONTOLOGY (SAFE + VALIDATED)
# ============================================================

import yaml
import json

# ============================================================
# CLEAN YAML (ALL TOKENS SAFE)
# ============================================================

SEMANTIC_YAML = """
version: 4.0
name: ARC_CLEAN_SEMANTIC_ONTOLOGY

relation_types:
  direct:
    weight: 1.0
  weak:
    weight: 0.3
  similar:
    weight: 0.2
  compositional:
    weight: 0.6
  structural:
    weight: 0.8

concepts:

  shape:
    direct_keywords: ["shape", "object", "figure", "form"]
    weak_keywords: ["structure", "component", "region"]

  polygon:
    parent: "shape"
    direct_keywords: ["polygon", "multi_sided"]

  three_dimensional_shape:
    direct_keywords: ["3d_object", "solid", "polyhedron"]

  color:
    direct_keywords: ["color", "pixel_value", "value"]

  pattern:
    direct_keywords: ["pattern", "repetition", "tiling"]

shapes_2d:

  triangle:
    sides: 3
    direct_keywords: ["triangle", "3_sides"]
    weak_keywords: ["pointed", "wedge"]
    relations:
      similar: ["pentagon", "square"]

  square:
    sides: 4
    direct_keywords: ["square"]
    weak_keywords: ["box", "block", "tile"]
    relations:
      similar: ["rectangle"]
      structural: ["grid", "tiling"]

  rectangle:
    sides: 4
    direct_keywords: ["rectangle"]
    weak_keywords: ["box", "frame"]
    relations:
      similar: ["square"]

  pentagon:
    sides: 5
    direct_keywords: ["pentagon", "5_sides"]
    weak_keywords: ["polygon"]
    relations:
      compositional:
        builds: ["dodecahedron"]

  hexagon:
    sides: 6
    direct_keywords: ["hexagon"]
    weak_keywords: ["honeycomb"]
    relations:
      structural: ["tiling"]

  circle_like:
    direct_keywords: ["circle", "round"]
    weak_keywords: ["blob", "disk"]

  line:
    direct_keywords: ["line"]
    weak_keywords: ["segment", "stroke"]

  frame:
    direct_keywords: ["frame", "border"]
    weak_keywords: ["outline", "edge"]
    relations:
      structural: ["boundary"]

shapes_3d:

  cube:
    direct_keywords: ["cube"]
    composed_of: ["square"]

  rectangular_prism:
    direct_keywords: ["cuboid"]
    composed_of: ["rectangle"]

  tetrahedron:
    direct_keywords: ["tetrahedron"]
    composed_of: ["triangle"]

  octahedron:
    direct_keywords: ["octahedron"]
    composed_of: ["triangle"]

  dodecahedron:
    direct_keywords: ["dodecahedron"]
    composed_of: ["pentagon"]
    activation_requirements:
      - "repeated_pentagon"
      - "closed_structure"

  icosahedron:
    direct_keywords: ["icosahedron"]
    composed_of: ["triangle"]
    activation_requirements:
      - "repeated_triangle"
      - "mesh_pattern"

color:

  types:
    primary:
      keywords: ["red", "blue", "yellow"]
    secondary:
      keywords: ["green", "purple", "orange"]

  properties:
    dominant:
      keywords: ["dominant_color"]
      requires: ["counting"]
    alternating:
      keywords: ["alternating_color"]
      requires: ["pattern"]
    uniform:
      keywords: ["uniform_color"]
      requires: ["identity"]
    boundary_color:
      keywords: ["edge_color"]
      requires: ["boundary"]

properties:

  closed:
    keywords: ["closed", "enclosed"]

  open:
    keywords: ["open"]

  symmetric:
    keywords: ["symmetric", "mirrored"]

  repeated:
    keywords: ["repeated", "tiled"]

  connected:
    keywords: ["connected", "touching"]

rules:

  extract_largest_object:
    triggers:
      direct: ["largest", "dominant"]
      weak: ["object", "cluster"]

  crop_nonzero_bbox:
    triggers:
      direct: ["crop", "bbox"]
      weak: ["region", "boundary"]

  color_map:
    triggers:
      direct: ["color_change"]
      weak: ["mapping"]

  tile_n:
    triggers:
      direct: ["repeat", "tile"]
      structural: ["pattern"]

  rotate_90:
    triggers:
      direct: ["rotate"]

  flip_horizontal:
    triggers:
      direct: ["mirror"]

pipelines:

  object_then_color:
    steps: ["extract_largest_object", "color_map"]

  crop_then_repeat:
    steps: ["crop_nonzero_bbox", "tile_n"]

  extract_then_rotate:
    steps: ["extract_largest_object", "rotate_90"]

activation_logic:

  rules:
    - name: "no_direct_2d_to_3d"
    - name: "3d_requires_structure"
    - name: "weak_never_promotes"
    - name: "compositional_requires_multiple_hits"

meta:
  goal: "eliminate semantic leakage"
"""

# ============================================================
# LOAD YAML SAFELY
# ============================================================

SEMANTIC_ONTOLOGY = yaml.safe_load(SEMANTIC_YAML)

# ============================================================
# BASIC VALIDATION
# ============================================================

def validate_ontology(ontology):
    assert "shapes_2d" in ontology, "Missing shapes_2d"
    assert "shapes_3d" in ontology, "Missing shapes_3d"
    assert "rules" in ontology, "Missing rules"
    print("✅ Ontology validation passed")

validate_ontology(SEMANTIC_ONTOLOGY)

# ============================================================
# MINIMAL SEMANTIC ENGINE (CLEAN VERSION)
# ============================================================

class SemanticEngine:

    def __init__(self, ontology):
        self.ontology = ontology

    def explain_keyword(self, keyword):
        keyword = keyword.lower()

        direct_2d = []
        direct_3d = []

        for name, data in self.ontology["shapes_2d"].items():
            if keyword in data.get("direct_keywords", []):
                direct_2d.append(name)

        for name, data in self.ontology["shapes_3d"].items():
            if keyword in data.get("direct_keywords", []):
                direct_3d.append(name)

        inferred_3d = []
        for shape in direct_2d:
            rel = self.ontology["shapes_2d"][shape].get("relations", {})
            if "compositional" in rel:
                inferred_3d += rel["compositional"].get("builds", [])

        return {
            "keyword": keyword,
            "direct_shapes_2d": direct_2d,
            "direct_shapes_3d": direct_3d,
            "inferred_shapes_3d": inferred_3d
        }

# ============================================================
# INIT ENGINE
# ============================================================

semantic_engine = SemanticEngine(SEMANTIC_ONTOLOGY)

# ============================================================
# TEST OUTPUT
# ============================================================

print("=" * 80)
print("CLEAN SEMANTIC SYSTEM LOADED")
print("=" * 80)

print("\nPentagon:")
print(json.dumps(semantic_engine.explain_keyword("pentagon"), indent=2))

print("\nBox:")
print(json.dumps(semantic_engine.explain_keyword("box"), indent=2))

✅ Ontology validation passed
CLEAN SEMANTIC SYSTEM LOADED

Pentagon:
{
  "keyword": "pentagon",
  "direct_shapes_2d": [
    "pentagon"
  ],
  "direct_shapes_3d": [],
  "inferred_shapes_3d": [
    "dodecahedron"
  ]
}

Box:
{
  "keyword": "box",
  "direct_shapes_2d": [],
  "direct_shapes_3d": [],
  "inferred_shapes_3d": []
}


In [19]:
# ============================================================
# ARC CLEAN SEMANTIC SYSTEM (FINAL — FULL CORRECT VERSION)
# ============================================================

import yaml
import json

# ============================================================
# CLEAN YAML ONTOLOGY
# ============================================================

SEMANTIC_YAML = """
version: 4.4
name: ARC_CLEAN_SEMANTIC_ONTOLOGY

relation_types:
  direct: {weight: 1.0}
  weak: {weight: 0.3}
  similar: {weight: 0.2}
  compositional: {weight: 0.6}
  structural: {weight: 0.8}

concepts:

  shape:
    direct_keywords: ["shape", "object", "figure", "form"]
    weak_keywords: ["structure", "component", "region"]

  polygon:
    parent: "shape"
    direct_keywords: ["polygon", "multi_sided"]

  three_dimensional_shape:
    direct_keywords: ["3d_object", "solid", "polyhedron"]

  color:
    direct_keywords: ["color", "pixel_value", "value"]

  pattern:
    direct_keywords: ["pattern", "repetition", "tiling"]

# ============================================================
# ALIASES
# ============================================================

aliases:

  box:
    maps_to:
      direct: ["square", "rectangle"]
      weak: ["frame", "region"]

  block:
    maps_to:
      direct: ["square"]
      weak: ["region"]

  outline:
    maps_to:
      direct: ["frame"]

  edge:
    maps_to:
      weak: ["line", "boundary"]

  blob:
    maps_to:
      weak: ["circle_like", "region"]

  tile:
    maps_to:
      direct: ["square"]
      structural: ["pattern"]

# ============================================================
# 2D SHAPES (WITH ROLES)
# ============================================================

shapes_2d:

  triangle:
    role: "solid"
    direct_keywords: ["triangle", "3_sides"]
    weak_keywords: ["pointed", "wedge"]

  square:
    role: "solid"
    direct_keywords: ["square"]
    weak_keywords: ["block", "tile"]

  rectangle:
    role: "solid"
    direct_keywords: ["rectangle"]
    weak_keywords: ["frame"]

  pentagon:
    role: "solid"
    direct_keywords: ["pentagon", "5_sides"]
    weak_keywords: ["polygon"]
    relations:
      compositional:
        builds: ["dodecahedron"]

  hexagon:
    role: "solid"
    direct_keywords: ["hexagon"]
    weak_keywords: ["honeycomb"]

  circle_like:
    role: "solid"
    direct_keywords: ["circle", "round"]
    weak_keywords: ["blob"]

  line:
    role: "edge"
    direct_keywords: ["line"]
    weak_keywords: ["segment"]

  frame:
    role: "boundary"
    direct_keywords: ["frame", "border"]
    weak_keywords: ["outline"]

# ============================================================
# 3D SHAPES
# ============================================================

shapes_3d:

  cube:
    direct_keywords: ["cube"]
    composed_of: ["square"]

  rectangular_prism:
    direct_keywords: ["cuboid"]
    composed_of: ["rectangle"]

  tetrahedron:
    direct_keywords: ["tetrahedron"]
    composed_of: ["triangle"]

  octahedron:
    direct_keywords: ["octahedron"]
    composed_of: ["triangle"]

  dodecahedron:
    direct_keywords: ["dodecahedron"]
    composed_of: ["pentagon"]
    activation_requirements: ["repeated_pentagon", "closed_structure"]

  icosahedron:
    direct_keywords: ["icosahedron"]
    composed_of: ["triangle"]

# ============================================================
# RULES
# ============================================================

rules:

  extract_largest_object:
    triggers:
      direct: ["largest", "dominant"]
      weak: ["object"]

  crop_nonzero_bbox:
    triggers:
      direct: ["crop"]
      weak: ["region", "boundary"]

  color_map:
    triggers:
      direct: ["color_change"]

  tile_n:
    triggers:
      direct: ["repeat", "tile"]

  rotate_90:
    triggers:
      direct: ["rotate"]

  flip_horizontal:
    triggers:
      direct: ["mirror"]

meta:
  goal: "clean semantic reasoning without leakage"
"""

# ============================================================
# LOAD + VALIDATE
# ============================================================

SEMANTIC_ONTOLOGY = yaml.safe_load(SEMANTIC_YAML)

def validate_ontology(ontology):
    assert "shapes_2d" in ontology
    assert "shapes_3d" in ontology
    assert "aliases" in ontology
    print("✅ Ontology validation passed")

validate_ontology(SEMANTIC_ONTOLOGY)

# ============================================================
# SEMANTIC ENGINE (FINAL CORRECT VERSION)
# ============================================================

class SemanticEngine:

    def __init__(self, ontology):
        self.ontology = ontology

    def resolve_alias(self, keyword):
        keyword = keyword.lower()
        alias_map = self.ontology.get("aliases", {})

        resolved = [{
            "term": keyword,
            "source": "direct"
        }]

        if keyword in alias_map:
            mapping = alias_map[keyword].get("maps_to", {})
            for k in ["direct", "weak", "structural"]:
                for item in mapping.get(k, []):
                    resolved.append({
                        "term": item,
                        "source": "alias"
                    })

        return resolved

    def explain_keyword(self, keyword):

        keyword = keyword.lower()
        resolved = self.resolve_alias(keyword)

        direct_2d = []
        direct_3d = []

        for entry in resolved:
            kw = entry["term"]
            source = entry["source"]

            for name, data in self.ontology["shapes_2d"].items():
                if kw in data.get("direct_keywords", []):
                    confidence = 1.0 if source == "direct" else 0.7
                    direct_2d.append({
                        "shape": name,
                        "role": data.get("role", "unknown"),
                        "confidence": confidence,
                        "source": source
                    })

            for name, data in self.ontology["shapes_3d"].items():
                if kw in data.get("direct_keywords", []):
                    confidence = 1.0 if source == "direct" else 0.7
                    direct_3d.append({
                        "shape": name,
                        "confidence": confidence,
                        "source": source
                    })

        # Deduplicate (keep highest confidence)
        def dedupe(items, key):
            best = {}
            for item in items:
                k = item[key]
                if k not in best or item["confidence"] > best[k]["confidence"]:
                    best[k] = item
            return list(best.values())

        direct_2d = dedupe(direct_2d, "shape")
        direct_3d = dedupe(direct_3d, "shape")

        # Compositional inference (2D → 3D)
        inferred_3d = set()
        for item in direct_2d:
            shape = item["shape"]
            rel = self.ontology["shapes_2d"][shape].get("relations", {})
            if "compositional" in rel:
                inferred_3d.update(rel["compositional"].get("builds", []))

        return {
            "keyword": keyword,
            "resolved_terms": resolved,
            "direct_shapes_2d": direct_2d,
            "direct_shapes_3d": direct_3d,
            "inferred_shapes_3d": sorted(list(inferred_3d))
        }

# ============================================================
# INIT ENGINE
# ============================================================

semantic_engine = SemanticEngine(SEMANTIC_ONTOLOGY)

# ============================================================
# TEST OUTPUT
# ============================================================

print("=" * 80)
print("FINAL SEMANTIC SYSTEM (CORRECT + COMPLETE) READY")
print("=" * 80)

print("\nPentagon:")
print(json.dumps(semantic_engine.explain_keyword("pentagon"), indent=2))

print("\nBox:")
print(json.dumps(semantic_engine.explain_keyword("box"), indent=2))

✅ Ontology validation passed
FINAL SEMANTIC SYSTEM (CORRECT + COMPLETE) READY

Pentagon:
{
  "keyword": "pentagon",
  "resolved_terms": [
    {
      "term": "pentagon",
      "source": "direct"
    }
  ],
  "direct_shapes_2d": [
    {
      "shape": "pentagon",
      "role": "solid",
      "confidence": 1.0,
      "source": "direct"
    }
  ],
  "direct_shapes_3d": [],
  "inferred_shapes_3d": [
    "dodecahedron"
  ]
}

Box:
{
  "keyword": "box",
  "resolved_terms": [
    {
      "term": "box",
      "source": "direct"
    },
    {
      "term": "square",
      "source": "alias"
    },
    {
      "term": "rectangle",
      "source": "alias"
    },
    {
      "term": "frame",
      "source": "alias"
    },
    {
      "term": "region",
      "source": "alias"
    }
  ],
  "direct_shapes_2d": [
    {
      "shape": "square",
      "role": "solid",
      "confidence": 0.7,
      "source": "alias"
    },
    {
      "shape": "rectangle",
      "role": "solid",
      "confidence": 0.7,
 

In [21]:
# ============================================================
# ARC CLEAN SEMANTIC SYSTEM + WIRED SOLVER LOOP
# FULL SINGLE COLAB CELL
# ============================================================

from __future__ import annotations

import copy
import json
import math
import re
import traceback
from dataclasses import dataclass, asdict, field
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple

# ------------------------------------------------------------
# YAML
# ------------------------------------------------------------
try:
    import yaml
except ImportError:
    !pip -q install pyyaml
    import yaml

# ============================================================
# CLEAN SEMANTIC ONTOLOGY
# ============================================================

SEMANTIC_YAML = """
version: 4.4
name: ARC_CLEAN_SEMANTIC_ONTOLOGY

relation_types:
  direct: {weight: 1.0}
  weak: {weight: 0.3}
  similar: {weight: 0.2}
  compositional: {weight: 0.6}
  structural: {weight: 0.8}

concepts:

  shape:
    direct_keywords: ["shape", "object", "figure", "form"]
    weak_keywords: ["structure", "component", "region"]

  polygon:
    parent: "shape"
    direct_keywords: ["polygon", "multi_sided"]

  three_dimensional_shape:
    direct_keywords: ["3d_object", "solid", "polyhedron"]

  color:
    direct_keywords: ["color", "pixel_value", "value"]

  pattern:
    direct_keywords: ["pattern", "repetition", "tiling"]

aliases:

  box:
    maps_to:
      direct: ["square", "rectangle"]
      weak: ["frame", "region"]

  block:
    maps_to:
      direct: ["square"]
      weak: ["region"]

  outline:
    maps_to:
      direct: ["frame"]

  edge:
    maps_to:
      weak: ["line", "boundary"]

  blob:
    maps_to:
      weak: ["circle_like", "region"]

  tile:
    maps_to:
      direct: ["square"]
      structural: ["pattern"]

shapes_2d:

  triangle:
    role: "solid"
    direct_keywords: ["triangle", "3_sides"]
    weak_keywords: ["pointed", "wedge"]

  square:
    role: "solid"
    direct_keywords: ["square"]
    weak_keywords: ["block", "tile"]

  rectangle:
    role: "solid"
    direct_keywords: ["rectangle"]
    weak_keywords: ["frame"]

  pentagon:
    role: "solid"
    direct_keywords: ["pentagon", "5_sides"]
    weak_keywords: ["polygon"]
    relations:
      compositional:
        builds: ["dodecahedron"]

  hexagon:
    role: "solid"
    direct_keywords: ["hexagon"]
    weak_keywords: ["honeycomb"]

  circle_like:
    role: "solid"
    direct_keywords: ["circle", "round"]
    weak_keywords: ["blob"]

  line:
    role: "edge"
    direct_keywords: ["line"]
    weak_keywords: ["segment"]

  frame:
    role: "boundary"
    direct_keywords: ["frame", "border"]
    weak_keywords: ["outline"]

shapes_3d:

  cube:
    direct_keywords: ["cube"]
    composed_of: ["square"]

  rectangular_prism:
    direct_keywords: ["cuboid"]
    composed_of: ["rectangle"]

  tetrahedron:
    direct_keywords: ["tetrahedron"]
    composed_of: ["triangle"]

  octahedron:
    direct_keywords: ["octahedron"]
    composed_of: ["triangle"]

  dodecahedron:
    direct_keywords: ["dodecahedron"]
    composed_of: ["pentagon"]
    activation_requirements: ["repeated_pentagon", "closed_structure"]

  icosahedron:
    direct_keywords: ["icosahedron"]
    composed_of: ["triangle"]

rules:

  extract_largest_object:
    triggers:
      direct: ["largest", "dominant"]
      weak: ["object"]

  crop_nonzero_bbox:
    triggers:
      direct: ["crop"]
      weak: ["region", "boundary"]

  color_map:
    triggers:
      direct: ["color_change"]

  tile_n:
    triggers:
      direct: ["repeat", "tile"]

  rotate_90:
    triggers:
      direct: ["rotate"]

  flip_horizontal:
    triggers:
      direct: ["mirror"]

meta:
  goal: "clean semantic reasoning without leakage"
"""

SEMANTIC_ONTOLOGY = yaml.safe_load(SEMANTIC_YAML)

def validate_ontology(ontology):
    assert "shapes_2d" in ontology
    assert "shapes_3d" in ontology
    assert "aliases" in ontology
    print("✅ Ontology validation passed")

validate_ontology(SEMANTIC_ONTOLOGY)


# ============================================================
# BASIC UTILITIES
# ============================================================

def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace("-", "_")
    text = re.sub(r"[^a-z0-9_\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str) -> List[str]:
    text = normalize_text(text)
    return text.split() if text else []

def generate_ngrams(tokens: List[str], max_n: int = 3) -> List[str]:
    grams = []
    n = len(tokens)
    for k in range(1, max_n + 1):
        for i in range(n - k + 1):
            grams.append("_".join(tokens[i:i+k]))
    return grams

def _safe_jsonable(x: Any) -> Any:
    try:
        json.dumps(x)
        return x
    except Exception:
        if isinstance(x, dict):
            return {str(k): _safe_jsonable(v) for k, v in x.items()}
        if isinstance(x, (list, tuple)):
            return [_safe_jsonable(v) for v in x]
        return repr(x)

def _deepcopy(x: Any) -> Any:
    try:
        return copy.deepcopy(x)
    except Exception:
        return x

def _clip01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))

def _safe_mean(values: List[float], default: float = 0.0) -> float:
    vals = [float(v) for v in values if isinstance(v, (int, float))]
    return sum(vals) / len(vals) if vals else default


# ============================================================
# SEMANTIC ENGINE
# ============================================================

class SemanticEngine:

    def __init__(self, ontology: Dict[str, Any]):
        self.ontology = ontology

    def resolve_alias(self, keyword: str) -> List[Dict[str, str]]:
        keyword = normalize_text(keyword)
        alias_map = self.ontology.get("aliases", {})

        resolved = [{
            "term": keyword,
            "source": "direct"
        }]

        if keyword in alias_map:
            mapping = alias_map[keyword].get("maps_to", {})
            for k in ["direct", "weak", "structural"]:
                for item in mapping.get(k, []):
                    resolved.append({
                        "term": normalize_text(item),
                        "source": "alias"
                    })

        # stable dedupe
        seen = set()
        out = []
        for item in resolved:
            key = (item["term"], item["source"])
            if key not in seen:
                seen.add(key)
                out.append(item)
        return out

    def explain_keyword(self, keyword: str) -> Dict[str, Any]:
        keyword = normalize_text(keyword)
        resolved = self.resolve_alias(keyword)

        direct_2d = []
        direct_3d = []

        for entry in resolved:
            kw = entry["term"]
            source = entry["source"]

            for name, data in self.ontology["shapes_2d"].items():
                if kw in [normalize_text(x) for x in data.get("direct_keywords", [])]:
                    confidence = 1.0 if source == "direct" else 0.7
                    direct_2d.append({
                        "shape": name,
                        "role": data.get("role", "unknown"),
                        "confidence": confidence,
                        "source": source
                    })

            for name, data in self.ontology["shapes_3d"].items():
                if kw in [normalize_text(x) for x in data.get("direct_keywords", [])]:
                    confidence = 1.0 if source == "direct" else 0.7
                    direct_3d.append({
                        "shape": name,
                        "confidence": confidence,
                        "source": source
                    })

        def dedupe(items: List[Dict[str, Any]], key: str) -> List[Dict[str, Any]]:
            best = {}
            for item in items:
                k = item[key]
                if k not in best or item["confidence"] > best[k]["confidence"]:
                    best[k] = item
            return list(best.values())

        direct_2d = dedupe(direct_2d, "shape")
        direct_3d = dedupe(direct_3d, "shape")

        inferred_3d = set()
        for item in direct_2d:
            shape = item["shape"]
            rel = self.ontology["shapes_2d"][shape].get("relations", {})
            if "compositional" in rel:
                inferred_3d.update(rel["compositional"].get("builds", []))

        return {
            "keyword": keyword,
            "resolved_terms": resolved,
            "direct_shapes_2d": direct_2d,
            "direct_shapes_3d": direct_3d,
            "inferred_shapes_3d": sorted(list(inferred_3d))
        }

    def analyze_text(self, text: str) -> Dict[str, Any]:
        tokens = tokenize(text)
        ngrams = generate_ngrams(tokens, max_n=3)

        shape_hits_2d = {}
        shape_hits_3d = {}
        inferred_3d = set()
        role_scores = defaultdict(float)
        term_details = []

        for term in sorted(set(ngrams)):
            info = self.explain_keyword(term)
            if info["direct_shapes_2d"] or info["direct_shapes_3d"] or info["inferred_shapes_3d"]:
                term_details.append(info)

            for item in info["direct_shapes_2d"]:
                s = item["shape"]
                if s not in shape_hits_2d or item["confidence"] > shape_hits_2d[s]["confidence"]:
                    shape_hits_2d[s] = item
                role_scores[item["role"]] += item["confidence"]

            for item in info["direct_shapes_3d"]:
                s = item["shape"]
                if s not in shape_hits_3d or item["confidence"] > shape_hits_3d[s]["confidence"]:
                    shape_hits_3d[s] = item

            inferred_3d.update(info["inferred_shapes_3d"])

        rule_hints = self.rule_hints_from_text(text, shape_hits_2d, role_scores)
        pipeline_hints = self.pipeline_hints_from_rules(rule_hints)

        return {
            "text": text,
            "tokens": tokens,
            "ngrams": ngrams,
            "shape_hints_2d": sorted(shape_hits_2d.values(), key=lambda x: (-x["confidence"], x["shape"])),
            "shape_hints_3d": sorted(shape_hits_3d.values(), key=lambda x: (-x["confidence"], x["shape"])),
            "inferred_shapes_3d": sorted(list(inferred_3d)),
            "role_scores": dict(role_scores),
            "rule_hints": rule_hints,
            "pipeline_hints": pipeline_hints,
            "term_details": term_details,
        }

    def rule_hints_from_text(self, text: str, shape_hits_2d: Dict[str, Dict[str, Any]], role_scores: Dict[str, float]) -> List[Dict[str, Any]]:
        text_norm = normalize_text(text)
        rules = self.ontology.get("rules", {})
        hints = []

        for rule_name, rule_data in rules.items():
            score = 0.0
            trigger_meta = {"direct": [], "weak": [], "structural": []}

            for trig_type, trig_list in rule_data.get("triggers", {}).items():
                for trig in trig_list:
                    trig_n = normalize_text(trig)
                    if trig_n in text_norm:
                        if trig_type == "direct":
                            score += 1.0
                        elif trig_type == "weak":
                            score += 0.35
                        elif trig_type == "structural":
                            score += 0.6
                        trigger_meta.setdefault(trig_type, []).append(trig_n)

            # role-based nudges
            if rule_name == "extract_largest_object" and role_scores.get("solid", 0.0) > 0:
                score += 0.25
            if rule_name == "crop_nonzero_bbox" and role_scores.get("boundary", 0.0) > 0:
                score += 0.25
            if rule_name == "tile_n" and ("repeat" in text_norm or "tile" in text_norm):
                score += 0.25
            if rule_name == "flip_horizontal" and "mirror" in text_norm:
                score += 0.25
            if rule_name == "rotate_90" and "rotate" in text_norm:
                score += 0.25

            if score > 0:
                hints.append({
                    "rule": rule_name,
                    "score": round(score, 4),
                    "matched_triggers": trigger_meta
                })

        hints.sort(key=lambda x: (-x["score"], x["rule"]))
        return hints

    def pipeline_hints_from_rules(self, rule_hints: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        rules_present = {x["rule"] for x in rule_hints}
        pipeline_library = {
            "object_then_color": ["extract_largest_object", "color_map"],
            "crop_then_repeat": ["crop_nonzero_bbox", "tile_n"],
            "extract_then_rotate": ["extract_largest_object", "rotate_90"],
            "extract_then_flip": ["extract_largest_object", "flip_horizontal"],
        }

        out = []
        for name, steps in pipeline_library.items():
            overlap = len(rules_present.intersection(steps))
            if overlap > 0:
                out.append({
                    "pipeline": name,
                    "steps": steps,
                    "score": overlap / len(steps)
                })
        out.sort(key=lambda x: (-x["score"], x["pipeline"]))
        return out


semantic_engine = SemanticEngine(SEMANTIC_ONTOLOGY)


# ============================================================
# ARC TASK UTILITIES
# ============================================================

def _is_grid(x: Any) -> bool:
    return (
        isinstance(x, list)
        and all(isinstance(r, list) for r in x)
        and all(all(isinstance(v, int) for v in r) for r in x)
    )

def _grid_shape(grid: List[List[int]]) -> Tuple[int, int]:
    if not _is_grid(grid):
        return (0, 0)
    return (len(grid), len(grid[0]) if grid else 0)

def _exact_match(a: List[List[int]], b: List[List[int]]) -> bool:
    return a == b

def _same_shape(a: List[List[int]], b: List[List[int]]) -> bool:
    return _grid_shape(a) == _grid_shape(b)

def _grid_colors(grid: List[List[int]]) -> List[int]:
    if not _is_grid(grid):
        return []
    return sorted({v for row in grid for v in row})

def _count_nonzero(grid: List[List[int]]) -> int:
    if not _is_grid(grid):
        return 0
    return sum(1 for row in grid for v in row if v != 0)

def _connected_components(grid: List[List[int]]) -> List[Dict[str, Any]]:
    if not _is_grid(grid) or not grid:
        return []
    h, w = len(grid), len(grid[0])
    seen = [[False] * w for _ in range(h)]
    comps = []

    for r in range(h):
        for c in range(w):
            if seen[r][c] or grid[r][c] == 0:
                continue
            color = grid[r][c]
            stack = [(r, c)]
            seen[r][c] = True
            cells = []

            while stack:
                rr, cc = stack.pop()
                cells.append((rr, cc))
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = rr + dr, cc + dc
                    if 0 <= nr < h and 0 <= nc < w and not seen[nr][nc] and grid[nr][nc] == color:
                        seen[nr][nc] = True
                        stack.append((nr, nc))

            rs = [x[0] for x in cells]
            cs = [x[1] for x in cells]
            comps.append({
                "color": color,
                "size": len(cells),
                "bbox": (min(rs), min(cs), max(rs), max(cs)),
            })
    return comps

def _infer_scale(inp: List[List[int]], out: List[List[int]]) -> Tuple[float, float]:
    ih, iw = _grid_shape(inp)
    oh, ow = _grid_shape(out)
    return (oh / ih if ih else 0.0, ow / iw if iw else 0.0)

def _looks_identity(rule_template: Any) -> bool:
    if isinstance(rule_template, str):
        s = rule_template.lower()
        return s in {"identity", "copy", "noop", "pass_through", "pass-through"}
    if isinstance(rule_template, (list, tuple)):
        joined = " ".join(map(str, rule_template)).lower()
        return "identity" in joined or joined in {"copy", "noop"}
    return False

def _has_composition(rule_template: Any) -> bool:
    return isinstance(rule_template, (list, tuple)) and len(rule_template) >= 2

def normalize_task(task: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(task, dict):
        raise TypeError("task must be a dict")
    train = task.get("train", [])
    test = task.get("test", [])
    if not isinstance(train, list) or not isinstance(test, list):
        raise ValueError("task must contain train/test lists")
    for i, pair in enumerate(train):
        if "input" not in pair or "output" not in pair:
            raise ValueError(f"train[{i}] missing input/output")
        if not _is_grid(pair["input"]) or not _is_grid(pair["output"]):
            raise ValueError(f"train[{i}] input/output must be integer grids")
    for i, pair in enumerate(test):
        if "input" not in pair or not _is_grid(pair["input"]):
            raise ValueError(f"test[{i}] missing valid input")
    return task


# ============================================================
# SEMANTIC SIGNATURE + MEMORY SIGNATURE
# ============================================================

def semantic_signature_from_text(task_description: str) -> Dict[str, Any]:
    analysis = semantic_engine.analyze_text(task_description or "")
    return {
        "shape_hints_2d": [
            {"shape": x["shape"], "role": x["role"], "confidence": x["confidence"]}
            for x in analysis["shape_hints_2d"][:6]
        ],
        "shape_hints_3d": [
            {"shape": x["shape"], "confidence": x["confidence"]}
            for x in analysis["shape_hints_3d"][:4]
        ],
        "inferred_shapes_3d": analysis["inferred_shapes_3d"][:4],
        "role_scores": analysis["role_scores"],
        "rule_hints": analysis["rule_hints"][:6],
        "pipeline_hints": analysis["pipeline_hints"][:4],
    }

def extract_task_signature(task: Dict[str, Any], semantic_signature: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    task = normalize_task(task)
    train = task["train"]
    test = task["test"]

    signature = {
        "n_train": len(train),
        "n_test": len(test),
        "train_input_shapes": [_grid_shape(p["input"]) for p in train],
        "train_output_shapes": [_grid_shape(p["output"]) for p in train],
        "train_input_colors": [_grid_colors(p["input"]) for p in train],
        "train_output_colors": [_grid_colors(p["output"]) for p in train],
        "train_input_nonzero": [_count_nonzero(p["input"]) for p in train],
        "train_output_nonzero": [_count_nonzero(p["output"]) for p in train],
        "train_input_object_counts": [len(_connected_components(p["input"])) for p in train],
        "train_output_object_counts": [len(_connected_components(p["output"])) for p in train],
        "train_scales": [_infer_scale(p["input"], p["output"]) for p in train],
        "test_input_shapes": [_grid_shape(p["input"]) for p in test],
        "semantic_signature": _safe_jsonable(semantic_signature or {}),
    }
    return signature


# ============================================================
# MEMORY
# ============================================================

def _default_memory_db() -> Dict[str, Any]:
    return {"entries": []}

def _get_memory_entries(memory_db: Optional[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if memory_db is None:
        return []
    if isinstance(memory_db, dict) and isinstance(memory_db.get("entries"), list):
        return memory_db["entries"]
    if isinstance(memory_db, list):
        return memory_db
    return []

def _jaccard(a: List[Any], b: List[Any]) -> float:
    sa = {json.dumps(_safe_jsonable(x), sort_keys=True) for x in a}
    sb = {json.dumps(_safe_jsonable(x), sort_keys=True) for x in b}
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def _numeric_list_sim(a: List[float], b: List[float]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    n = min(len(a), len(b))
    sims = []
    for i in range(n):
        av, bv = float(a[i]), float(b[i])
        denom = max(abs(av), abs(bv), 1.0)
        sims.append(1.0 - min(abs(av - bv) / denom, 1.0))
    return _safe_mean(sims, 0.0)

def _flatten_pairs(xs: List[Tuple[float, float]]) -> List[float]:
    out = []
    for a, b in xs:
        out.extend([a, b])
    return out

def _shape_list_sim(a: List[Tuple[int, int]], b: List[Tuple[int, int]]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    n = min(len(a), len(b))
    exact = sum(1 for i in range(n) if tuple(a[i]) == tuple(b[i]))
    return exact / max(len(a), len(b))

def signature_similarity(sig_a: Dict[str, Any], sig_b: Dict[str, Any]) -> float:
    pieces = [
        _shape_list_sim(sig_a.get("train_input_shapes", []), sig_b.get("train_input_shapes", [])),
        _shape_list_sim(sig_a.get("train_output_shapes", []), sig_b.get("train_output_shapes", [])),
        _numeric_list_sim(sig_a.get("train_input_nonzero", []), sig_b.get("train_input_nonzero", [])),
        _numeric_list_sim(sig_a.get("train_output_nonzero", []), sig_b.get("train_output_nonzero", [])),
        _numeric_list_sim(sig_a.get("train_input_object_counts", []), sig_b.get("train_input_object_counts", [])),
        _numeric_list_sim(sig_a.get("train_output_object_counts", []), sig_b.get("train_output_object_counts", [])),
        _numeric_list_sim(_flatten_pairs(sig_a.get("train_scales", [])), _flatten_pairs(sig_b.get("train_scales", []))),
        _jaccard(sig_a.get("train_input_colors", []), sig_b.get("train_input_colors", [])),
        _jaccard(sig_a.get("train_output_colors", []), sig_b.get("train_output_colors", [])),
    ]

    sem_a = sig_a.get("semantic_signature", {})
    sem_b = sig_b.get("semantic_signature", {})

    if sem_a or sem_b:
        pieces.extend([
            _jaccard(sem_a.get("shape_hints_2d", []), sem_b.get("shape_hints_2d", [])),
            _jaccard(sem_a.get("shape_hints_3d", []), sem_b.get("shape_hints_3d", [])),
            _jaccard(sem_a.get("inferred_shapes_3d", []), sem_b.get("inferred_shapes_3d", [])),
            _jaccard(sem_a.get("rule_hints", []), sem_b.get("rule_hints", [])),
            _jaccard(sem_a.get("pipeline_hints", []), sem_b.get("pipeline_hints", [])),
        ])

    return _clip01(_safe_mean(pieces, 0.0))

def retrieve_from_memory(memory_db: Optional[Dict[str, Any]], signature: Dict[str, Any], threshold: float = 0.85, debug: bool = True) -> Dict[str, Any]:
    best_entry, best_score = None, -1.0

    for entry in _get_memory_entries(memory_db):
        try:
            score = signature_similarity(signature, entry.get("signature", {}))
            if score > best_score:
                best_score = score
                best_entry = entry
        except Exception:
            continue

    hit = best_entry is not None and best_score >= threshold
    if debug:
        print(f"[MEMORY] {'HIT' if hit else 'MISS'} | best_similarity={max(best_score, 0.0):.4f} | threshold={threshold:.4f}")

    return {
        "hit": hit,
        "similarity": max(best_score, 0.0),
        "entry": _deepcopy(best_entry)
    }

def write_to_memory_db(memory_db: Optional[Dict[str, Any]], signature: Dict[str, Any], best: Any, task_id: Optional[str], debug: bool = True) -> Dict[str, Any]:
    db = _deepcopy(memory_db) if memory_db is not None else _default_memory_db()
    if isinstance(db, list):
        db = {"entries": db}
    db.setdefault("entries", [])
    db["entries"].append({
        "task_id": task_id,
        "signature": _safe_jsonable(signature),
        "rule_template": _safe_jsonable(best.rule_template),
        "parameters": _safe_jsonable(best.parameters),
        "score": best.controller_score if best.controller_score else best.avg_score,
        "avg_score": best.avg_score,
        "exact_all": best.exact_all,
        "explanation": best.explanation,
    })
    if debug:
        print(f"[MEMORY WRITE] stored task_id={task_id} rule={best.rule_template}")
    return db


# ============================================================
# EXECUTION + SCORING
# ============================================================

def execute_pipeline_on_grid(rule_template: Any, parameters: Dict[str, Any], grid: List[List[int]]) -> List[List[int]]:
    fn = globals().get("execute_compositional_grammar")
    if callable(fn):
        for call in [
            lambda: fn({"rule_template": rule_template, "parameters": parameters}, grid),
            lambda: fn(rule_template, grid, parameters=parameters),
            lambda: fn(rule_template, parameters, grid),
        ]:
            try:
                return call()
            except Exception:
                pass

    fn = globals().get("apply_pipeline")
    if callable(fn):
        try:
            return fn(grid, rule_template, parameters)
        except Exception:
            pass

    fn = globals().get("execute_pipeline")
    if callable(fn):
        try:
            return fn(rule_template, parameters, grid)
        except Exception:
            pass

    if _looks_identity(rule_template):
        return _deepcopy(grid)

    raise RuntimeError("No compatible pipeline execution function found.")

def score_prediction(pred: List[List[int]], target: List[List[int]]) -> float:
    if _exact_match(pred, target):
        return 1.0
    if not _same_shape(pred, target):
        return 0.0
    total = len(pred) * len(pred[0]) if pred and pred[0] else 0
    if total == 0:
        return 0.0
    match = sum(1 for r in range(len(pred)) for c in range(len(pred[0])) if pred[r][c] == target[r][c])
    return match / total


# ============================================================
# CANDIDATES
# ============================================================

@dataclass
class Candidate:
    rule_template: Any
    parameters: Dict[str, Any] = field(default_factory=dict)
    avg_score: float = 0.0
    exact_all: bool = False
    exact_count: int = 0
    pair_scores: List[float] = field(default_factory=list)
    controller_score: float = 0.0
    source: str = "reasoning"
    explanation: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self):
        return asdict(self)

def _normalize_candidate(raw: Any) -> Candidate:
    if isinstance(raw, Candidate):
        return raw
    if isinstance(raw, dict):
        return Candidate(
            rule_template=raw.get("rule_template", raw.get("pipeline", raw.get("rule"))),
            parameters=_deepcopy(raw.get("parameters", {})) or {},
            source=raw.get("source", "reasoning"),
            explanation=raw.get("explanation", ""),
            metadata=_deepcopy(raw.get("metadata", {})) or {},
        )
    if isinstance(raw, (str, list, tuple)):
        return Candidate(rule_template=_deepcopy(raw), parameters={}, source="reasoning")
    raise TypeError(f"Unsupported candidate type: {type(raw)}")

def semantic_seed_candidates(semantic_analysis: Dict[str, Any]) -> List[Candidate]:
    candidates = []

    # rules from semantic hints
    for hint in semantic_analysis.get("rule_hints", [])[:8]:
        candidates.append(Candidate(
            rule_template=[hint["rule"]],
            source="semantic_seed",
            explanation=f"Seeded from semantic rule hint: {hint['rule']}",
            metadata={"semantic_rule_hint": hint}
        ))

    # pipelines from semantic hints
    for hint in semantic_analysis.get("pipeline_hints", [])[:5]:
        candidates.append(Candidate(
            rule_template=hint["steps"],
            source="semantic_seed",
            explanation=f"Seeded from semantic pipeline hint: {hint['pipeline']}",
            metadata={"semantic_pipeline_hint": hint}
        ))

    # role-based seeds
    role_scores = semantic_analysis.get("role_scores", {})
    if role_scores.get("boundary", 0.0) > 0:
        candidates.append(Candidate(
            rule_template=["crop_nonzero_bbox"],
            source="semantic_seed",
            explanation="Boundary role suggested bbox crop.",
            metadata={"semantic_role": "boundary"}
        ))
        candidates.append(Candidate(
            rule_template=["draw_border"],
            source="semantic_seed",
            explanation="Boundary role suggested border reasoning.",
            metadata={"semantic_role": "boundary"}
        ))

    if role_scores.get("solid", 0.0) > 0:
        candidates.append(Candidate(
            rule_template=["extract_largest_object"],
            source="semantic_seed",
            explanation="Solid role suggested object extraction.",
            metadata={"semantic_role": "solid"}
        ))

    return candidates

def generate_candidate_pipelines(task: Dict[str, Any], semantic_analysis: Optional[Dict[str, Any]] = None, candidate_limit: int = 256, debug: bool = True) -> List[Candidate]:
    raw_candidates = None

    for name in ["generate_candidate_pipelines", "generate_pipelines", "pipeline_generator"]:
        fn = globals().get(name)
        if callable(fn) and fn is not generate_candidate_pipelines:
            try:
                raw_candidates = fn(task)
                break
            except Exception:
                continue

    candidates = []
    if raw_candidates is not None:
        candidates = [_normalize_candidate(x) for x in raw_candidates]
    else:
        seed_templates = [
            "identity",
            ["extract_largest_object"],
            ["crop_nonzero_bbox"],
            ["color_map"],
            ["tile_n"],
            ["rotate_90"],
            ["flip_horizontal"],
            ["extract_largest_object", "color_map"],
            ["crop_nonzero_bbox", "tile_n"],
            ["extract_largest_object", "rotate_90"],
            ["extract_largest_object", "flip_horizontal"],
        ]
        candidates = [Candidate(rule_template=t) for t in seed_templates]

    if semantic_analysis is not None:
        candidates.extend(semantic_seed_candidates(semantic_analysis))

    # dedupe
    seen = set()
    out = []
    for cand in candidates:
        key = json.dumps({
            "rule_template": _safe_jsonable(cand.rule_template),
            "parameters": _safe_jsonable(cand.parameters),
        }, sort_keys=True)
        if key not in seen:
            seen.add(key)
            out.append(cand)

    out = out[:candidate_limit]
    if debug:
        print(f"[CANDIDATES] generated={len(out)}")
    return out


# ============================================================
# EVALUATION
# ============================================================

def evaluate_candidate_on_train(task: Dict[str, Any], candidate: Candidate) -> Candidate:
    out = _deepcopy(candidate)
    out.pair_scores = []

    for pair in task["train"]:
        try:
            pred = execute_pipeline_on_grid(out.rule_template, out.parameters, pair["input"])
            out.pair_scores.append(score_prediction(pred, pair["output"]))
        except Exception:
            out.pair_scores.append(0.0)

    out.avg_score = _safe_mean(out.pair_scores, 0.0)
    out.exact_count = sum(1 for s in out.pair_scores if s == 1.0)
    out.exact_all = (out.exact_count == len(task["train"])) if task["train"] else False

    if not out.explanation:
        out.explanation = f"avg_score={out.avg_score:.4f}, exact_count={out.exact_count}/{len(task['train'])}"

    return out

def score_candidates(task: Dict[str, Any], candidates: List[Candidate], debug: bool = True) -> List[Candidate]:
    scored = [evaluate_candidate_on_train(task, c) for c in candidates]
    scored.sort(
        key=lambda c: (
            c.exact_all,
            c.exact_count,
            c.avg_score,
            not _looks_identity(c.rule_template),
            _has_composition(c.rule_template),
        ),
        reverse=True
    )
    if debug:
        print("[SCORING] top candidates:")
        for i, cand in enumerate(scored[:10]):
            print(f"  {i+1:02d}. rule={cand.rule_template} | avg={cand.avg_score:.4f} | exact_all={cand.exact_all} | exact_count={cand.exact_count}")
    return scored


# ============================================================
# SEMANTIC ALIGNMENT
# ============================================================

def _flatten_rule_template(rule_template: Any) -> List[str]:
    if isinstance(rule_template, str):
        return [normalize_text(rule_template)]
    if isinstance(rule_template, (list, tuple)):
        return [normalize_text(str(x)) for x in rule_template]
    return [normalize_text(str(rule_template))]

def semantic_alignment_score(candidate: Candidate, semantic_analysis: Optional[Dict[str, Any]]) -> Dict[str, float]:
    if semantic_analysis is None:
        return {
            "rule_align": 0.0,
            "pipeline_align": 0.0,
            "role_align": 0.0,
            "shape_align": 0.0,
            "total": 0.0,
        }

    flat = _flatten_rule_template(candidate.rule_template)

    hinted_rules = [normalize_text(x["rule"]) for x in semantic_analysis.get("rule_hints", [])]
    hinted_pipes = [[normalize_text(s) for s in x["steps"]] for x in semantic_analysis.get("pipeline_hints", [])]
    shape_roles = semantic_analysis.get("role_scores", {})
    shape_names_2d = [normalize_text(x["shape"]) for x in semantic_analysis.get("shape_hints_2d", [])]

    rule_align = _jaccard(flat, hinted_rules)

    pipeline_align = 0.0
    for pipe in hinted_pipes:
        pipeline_align = max(pipeline_align, _jaccard(flat, pipe))

    role_align = 0.0
    if "crop_nonzero_bbox" in flat or "draw_border" in flat:
        role_align += min(shape_roles.get("boundary", 0.0), 1.0)
    if "extract_largest_object" in flat:
        role_align += min(shape_roles.get("solid", 0.0), 1.0)
    if "tile_n" in flat and "pattern" in normalize_text(semantic_analysis.get("text", "")):
        role_align += 0.5
    role_align = min(role_align, 1.0)

    shape_align = 0.0
    if "extract_largest_object" in flat and shape_names_2d:
        shape_align += 0.25
    if "crop_nonzero_bbox" in flat and any(x["role"] == "boundary" for x in semantic_analysis.get("shape_hints_2d", [])):
        shape_align += 0.5
    shape_align = min(shape_align, 1.0)

    total = rule_align + pipeline_align + role_align + shape_align
    return {
        "rule_align": rule_align,
        "pipeline_align": pipeline_align,
        "role_align": role_align,
        "shape_align": shape_align,
        "total": total,
    }


# ============================================================
# CONTROLLER
# ============================================================

DEFAULT_SOLVER_CONFIG = {
    "memory_threshold": 0.85,
    "memory_write_threshold": 0.98,
    "candidate_limit": 256,
    "debug": True,
    "exact_bonus": 0.25,
    "compositional_bonus": 0.05,
    "semantic_rule_bonus": 0.10,
    "semantic_pipeline_bonus": 0.10,
    "semantic_role_bonus": 0.08,
    "semantic_shape_bonus": 0.05,
    "identity_penalty": 0.08,
}

def _controller_adjusted_score(candidate: Candidate, semantic_analysis: Optional[Dict[str, Any]], config: Dict[str, Any]) -> Candidate:
    score = float(candidate.avg_score)

    if candidate.exact_all:
        score += config["exact_bonus"]

    if _has_composition(candidate.rule_template):
        score += config["compositional_bonus"]

    sem = semantic_alignment_score(candidate, semantic_analysis)
    score += config["semantic_rule_bonus"] * sem["rule_align"]
    score += config["semantic_pipeline_bonus"] * sem["pipeline_align"]
    score += config["semantic_role_bonus"] * sem["role_align"]
    score += config["semantic_shape_bonus"] * sem["shape_align"]

    if _looks_identity(candidate.rule_template) and not candidate.exact_all:
        score -= config["identity_penalty"]

    candidate.controller_score = score
    candidate.metadata = candidate.metadata or {}
    candidate.metadata["semantic_alignment"] = sem
    return candidate

def grammar_controller_select(task: Dict[str, Any], candidates: List[Candidate], semantic_analysis: Optional[Dict[str, Any]], config: Dict[str, Any], debug: bool = True) -> Candidate:
    external = globals().get("grammar_controller")
    if callable(external):
        try:
            raw = external(task, [c.to_dict() for c in candidates])
            best = _normalize_candidate(raw)

            matched = None
            for c in candidates:
                if _safe_jsonable(c.rule_template) == _safe_jsonable(best.rule_template) and _safe_jsonable(c.parameters) == _safe_jsonable(best.parameters):
                    matched = c
                    break

            if matched is not None:
                best.avg_score = matched.avg_score
                best.exact_all = matched.exact_all
                best.exact_count = matched.exact_count
                best.pair_scores = matched.pair_scores
                if not best.explanation:
                    best.explanation = matched.explanation

            best = _controller_adjusted_score(best, semantic_analysis, config)
            if debug:
                print(f"[CONTROLLER] external decision | rule={best.rule_template} | avg={best.avg_score:.4f} | controller_score={best.controller_score:.4f}")
            return best
        except Exception as e:
            if debug:
                print(f"[CONTROLLER] external controller failed, using fallback: {e}")

    ranked = []
    for c in candidates:
        ranked.append(_controller_adjusted_score(_deepcopy(c), semantic_analysis, config))

    ranked.sort(key=lambda c: (c.exact_all, c.controller_score, c.exact_count, c.avg_score), reverse=True)
    best = ranked[0]

    if debug:
        print(f"[CONTROLLER] fallback decision | rule={best.rule_template} | avg={best.avg_score:.4f} | controller_score={best.controller_score:.4f}")

    return best


# ============================================================
# TRACE
# ============================================================

def build_solver_trace(task_id: Optional[str], task: Dict[str, Any], signature: Dict[str, Any], semantic_analysis: Dict[str, Any], mode: str, memory_result: Dict[str, Any], candidates: List[Candidate], best: Candidate) -> Dict[str, Any]:
    ext = globals().get("build_compositional_trace")
    if callable(ext):
        try:
            trace = ext(task_id, task, {
                "rule_template": best.rule_template,
                "parameters": best.parameters,
                "score": best.controller_score if best.controller_score else best.avg_score,
                "avg_score": best.avg_score,
                "exact_all": best.exact_all,
                "explanation": best.explanation,
            })
            if isinstance(trace, dict):
                trace["solver_mode"] = mode
                trace["memory_similarity"] = memory_result.get("similarity", 0.0)
                trace["semantic_analysis"] = _safe_jsonable(semantic_analysis)
                return trace
        except Exception:
            pass

    return {
        "task_id": task_id,
        "solver_mode": mode,
        "signature": _safe_jsonable(signature),
        "semantic_analysis": _safe_jsonable(semantic_analysis),
        "memory": {
            "hit": memory_result.get("hit", False),
            "similarity": memory_result.get("similarity", 0.0),
            "matched_task_id": (memory_result.get("entry", {}) or {}).get("task_id"),
        },
        "candidate_summary": [
            {
                "rule_template": _safe_jsonable(c.rule_template),
                "parameters": _safe_jsonable(c.parameters),
                "avg_score": c.avg_score,
                "exact_all": c.exact_all,
                "exact_count": c.exact_count,
                "controller_score": c.controller_score,
                "semantic_alignment": c.metadata.get("semantic_alignment", {}),
                "source": c.source,
            }
            for c in candidates[:20]
        ],
        "selected": {
            "rule_template": _safe_jsonable(best.rule_template),
            "parameters": _safe_jsonable(best.parameters),
            "avg_score": best.avg_score,
            "exact_all": best.exact_all,
            "controller_score": best.controller_score,
            "semantic_alignment": best.metadata.get("semantic_alignment", {}),
            "explanation": best.explanation,
        },
    }


# ============================================================
# MAIN SOLVER
# ============================================================

def execute_selected_pipeline_on_tests(task: Dict[str, Any], best: Candidate) -> List[List[List[int]]]:
    outputs = []
    for pair in task["test"]:
        outputs.append(execute_pipeline_on_grid(best.rule_template, best.parameters, pair["input"]))
    return outputs

def solve_task(
    task: Dict[str, Any],
    task_id: Optional[str] = None,
    task_description: str = "",
    memory_db: Optional[Dict[str, Any]] = None,
    config: Optional[Dict[str, Any]] = None,
    write_memory: bool = False,
    return_updated_memory: bool = False,
) -> Dict[str, Any]:
    cfg = dict(DEFAULT_SOLVER_CONFIG)
    if config:
        cfg.update(config)

    debug = bool(cfg["debug"])

    try:
        task = normalize_task(task)

        # 1. semantic analysis first
        semantic_analysis = semantic_engine.analyze_text(task_description or "")
        semantic_signature = semantic_signature_from_text(task_description or "")

        if debug:
            print("[SEMANTICS] roles:", semantic_analysis.get("role_scores", {}))
            print("[SEMANTICS] rule hints:", [x["rule"] for x in semantic_analysis.get("rule_hints", [])[:5]])
            print("[SEMANTICS] pipeline hints:", [x["pipeline"] for x in semantic_analysis.get("pipeline_hints", [])[:3]])
            print("[SEMANTICS] 2D shapes:", [x["shape"] for x in semantic_analysis.get("shape_hints_2d", [])[:5]])
            print("[SEMANTICS] inferred 3D:", semantic_analysis.get("inferred_shapes_3d", [])[:5])

        # 2. task signature with semantics
        signature = extract_task_signature(task, semantic_signature=semantic_signature)

        # 3. memory retrieval
        memory_result = retrieve_from_memory(
            memory_db=memory_db,
            signature=signature,
            threshold=float(cfg["memory_threshold"]),
            debug=debug,
        )

        mode = "reasoning"
        candidates: List[Candidate] = []

        # 4. memory path
        if memory_result["hit"]:
            entry = memory_result["entry"]
            best = Candidate(
                rule_template=entry.get("rule_template"),
                parameters=_deepcopy(entry.get("parameters", {})) or {},
                source="memory",
                explanation=entry.get("explanation", f"Retrieved from memory with similarity={memory_result['similarity']:.4f}"),
                metadata={"memory_similarity": memory_result["similarity"]},
            )
            best = evaluate_candidate_on_train(task, best)
            best = _controller_adjusted_score(best, semantic_analysis, cfg)
            candidates = [best]
            mode = "memory"
            if debug:
                print(f"[SOLVER] mode=memory | rule={best.rule_template}")

        # 5. reasoning path
        else:
            candidates = generate_candidate_pipelines(
                task=task,
                semantic_analysis=semantic_analysis,
                candidate_limit=int(cfg["candidate_limit"]),
                debug=debug,
            )
            candidates = score_candidates(task, candidates, debug=debug)
            best = grammar_controller_select(task, candidates, semantic_analysis, cfg, debug=debug)
            if debug:
                print(f"[SOLVER] mode=reasoning | selected={best.rule_template}")

        # 6. trace
        trace = build_solver_trace(task_id, task, signature, semantic_analysis, mode, memory_result, candidates, best)

        # 7. explicit memory write
        updated_memory_db = memory_db
        should_write = write_memory and best.avg_score >= float(cfg["memory_write_threshold"]) and not _looks_identity(best.rule_template)
        if should_write:
            updated_memory_db = write_to_memory_db(memory_db, signature, best, task_id, debug=debug)
        elif debug:
            print(f"[MEMORY WRITE] skipped | write_memory={write_memory} | avg_score={best.avg_score:.4f} | identity={_looks_identity(best.rule_template)}")

        # 8. test execution
        test_output = execute_selected_pipeline_on_tests(task, best)

        result = {
            "task_id": task_id,
            "mode": mode,
            "pipeline": _safe_jsonable(best.rule_template),
            "parameters": _safe_jsonable(best.parameters),
            "score": float(best.controller_score if best.controller_score else best.avg_score),
            "avg_score": float(best.avg_score),
            "exact_all": bool(best.exact_all),
            "explanation": best.explanation,
            "semantic_analysis": _safe_jsonable(semantic_analysis),
            "trace": _safe_jsonable(trace),
            "test_output": _safe_jsonable(test_output),
        }

        if return_updated_memory:
            result["updated_memory_db"] = _safe_jsonable(updated_memory_db)

        return result

    except Exception as e:
        err = traceback.format_exc()
        print("[SOLVER ERROR]")
        print(err)
        return {
            "task_id": task_id,
            "mode": "error",
            "pipeline": None,
            "parameters": {},
            "score": 0.0,
            "avg_score": 0.0,
            "exact_all": False,
            "explanation": f"Solver failed: {type(e).__name__}: {e}",
            "semantic_analysis": {},
            "trace": {
                "task_id": task_id,
                "solver_mode": "error",
                "error_type": type(e).__name__,
                "error_message": str(e),
                "traceback": err,
            },
            "test_output": [],
            **({"updated_memory_db": _safe_jsonable(memory_db)} if return_updated_memory else {})
        }


# ============================================================
# DATASET SOLVER
# ============================================================

def solve_dataset(
    tasks: Dict[str, Dict[str, Any]],
    descriptions: Optional[Dict[str, str]] = None,
    memory_db: Optional[Dict[str, Any]] = None,
    config: Optional[Dict[str, Any]] = None,
    write_memory: bool = False,
) -> Dict[str, Any]:
    cfg = dict(DEFAULT_SOLVER_CONFIG)
    if config:
        cfg.update(config)

    debug = bool(cfg["debug"])
    current_memory = _deepcopy(memory_db) if memory_db is not None else _default_memory_db()
    descriptions = descriptions or {}

    results_by_task = {}
    n_total = 0
    n_memory = 0
    n_reasoning = 0
    n_error = 0
    train_exact_all = 0

    for task_id, task in tasks.items():
        result = solve_task(
            task=task,
            task_id=task_id,
            task_description=descriptions.get(task_id, ""),
            memory_db=current_memory,
            config=cfg,
            write_memory=write_memory,
            return_updated_memory=True,
        )
        results_by_task[task_id] = result
        current_memory = result.get("updated_memory_db", current_memory)

        n_total += 1
        mode = result.get("mode", "error")
        if mode == "memory":
            n_memory += 1
        elif mode == "reasoning":
            n_reasoning += 1
        else:
            n_error += 1

        if result.get("exact_all", False):
            train_exact_all += 1

    summary = {
        "n_tasks": n_total,
        "train_exact_all_count": train_exact_all,
        "train_exact_all_rate": (train_exact_all / n_total) if n_total else 0.0,
        "memory_usage_count": n_memory,
        "memory_usage_rate": (n_memory / n_total) if n_total else 0.0,
        "reasoning_count": n_reasoning,
        "reasoning_rate": (n_reasoning / n_total) if n_total else 0.0,
        "error_count": n_error,
        "error_rate": (n_error / n_total) if n_total else 0.0,
    }

    if debug:
        print("[DATASET SUMMARY]")
        for k, v in summary.items():
            print(f"  {k}: {v}")

    return {
        "results_by_task": results_by_task,
        "summary": summary,
        "updated_memory_db": current_memory,
    }


# ============================================================
# OPTIONAL ARC-AGI-2 SUBMISSION FORMAT HELPER
# ============================================================

def make_arc_agi_2_submission(results_by_task: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    submission = {}
    for task_id, result in results_by_task.items():
        test_outputs = result.get("test_output", [])
        submission[task_id] = []
        for grid in test_outputs:
            submission[task_id].append({
                "attempt_1": grid,
                "attempt_2": grid,
            })
    return submission


# ============================================================
# QUICK SANITY PRINTS
# ============================================================

print("=" * 88)
print("ARC SEMANTIC SYSTEM + SOLVER LOOP WIRED")
print("=" * 88)

print("\nSemantic check: pentagon")
print(json.dumps(semantic_engine.explain_keyword("pentagon"), indent=2))

print("\nSemantic check: box")
print(json.dumps(semantic_engine.explain_keyword("box"), indent=2))

print("\nReady to use:")
print("  result = solve_task(task, task_id='abc123', task_description='boxed region with repeated pattern')")
print("  dataset_result = solve_dataset(train_task_map, descriptions={})")
print("  submission = make_arc_agi_2_submission(dataset_result['results_by_task'])")

✅ Ontology validation passed
ARC SEMANTIC SYSTEM + SOLVER LOOP WIRED

Semantic check: pentagon
{
  "keyword": "pentagon",
  "resolved_terms": [
    {
      "term": "pentagon",
      "source": "direct"
    }
  ],
  "direct_shapes_2d": [
    {
      "shape": "pentagon",
      "role": "solid",
      "confidence": 1.0,
      "source": "direct"
    }
  ],
  "direct_shapes_3d": [],
  "inferred_shapes_3d": [
    "dodecahedron"
  ]
}

Semantic check: box
{
  "keyword": "box",
  "resolved_terms": [
    {
      "term": "box",
      "source": "direct"
    },
    {
      "term": "square",
      "source": "alias"
    },
    {
      "term": "rectangle",
      "source": "alias"
    },
    {
      "term": "frame",
      "source": "alias"
    },
    {
      "term": "region",
      "source": "alias"
    }
  ],
  "direct_shapes_2d": [
    {
      "shape": "square",
      "role": "solid",
      "confidence": 0.7,
      "source": "alias"
    },
    {
      "shape": "rectangle",
      "role": "solid",
     